# Assessment 2: Refreshable Multi-Source Cyber Threat Intelligence (CTI) RAG Assistant

**Base model retained:** Google Gemma 2 2B Instruct (4-bit NF4)  
**Retrieval:** ChromaDB + BM25 hybrid retrieval  
**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`  
**Application:** CTI research and preliminary phishing-email triage

## 1. Introduction & Problem Statement
Cybersecurity analysts need to investigate both historical threat behaviour and more recent attacks, vulnerabilities and phishing infrastructure. Information is spread across incident summaries, CVE records, ransomware listings, CTI guidance and historical reports. Manually correlating these sources can slow triage and incident response.

**Target users:** SOC analysts, threat hunters, incident responders and cybersecurity researchers.

**Goal:** Build a multi-source Retrieval-Augmented Generation (RAG) assistant using the existing open-weights Gemma 2 2B Instruct model. The application retrieves relevant evidence, provides source/date context and generates concise answers for CTI research and preliminary phishing-email triage.

## Scope of This Project

This project follows Pathway A: Retrieval-Augmented Generation (RAG). It keeps the existing Gemma 2 2B model and improves the evidence collection and retrieval design.

1. **Problem:** Analysts need to find and correlate information across historical reports, incident summaries, vulnerability records and phishing indicators.
2. **Solution:** Use Chroma/BM25 retrieval for narrative CTI knowledge, structured lookups for CVE and ransomware records, and exact blocklist matching for email URLs.
3. **Boundary:** This is analyst decision support, not a malware sandbox, mail gateway or comprehensive live threat-intelligence platform. A blocklist match is a lead; a non-match does not establish safety. A human analyst must verify important findings.

## 2. Data sources and freshness strategy

This notebook uses a **multi-source RAG design**. It does not assume that one dataset contains every type of CTI evidence.

| Source | Purpose | Refresh / limitation |
|---|---|---|
| `threatcluster/threat-incident-clusters` | Recent incident summaries, entities, reporting dates and source outlets | Hugging Face snapshot generated 2026-09-06; periodic snapshot, not live. Summaries are model-generated and need corroboration. |
| `threatcluster/cve-exploitation-signals` | CVE descriptions, CVSS, affected products, CISA KEV status, exploit availability and ransomware-use field | Snapshot generated 2026-09-06; structured lookup, not a live vulnerability feed. |
| `threatcluster/ransomware-leak-site-victims` | Ransomware group claims, named organisations, sectors, countries and dates | Snapshot generated 2026-09-06. A leak-site listing is a criminal group's claim, not proof of a confirmed breach. |
| `reloading0101/threat-intelligence-dataset` | Curated CTI Q&A covering vulnerabilities, ATT&CK, email threats and incident response | Curated instruction/Q&A data grounded in public sources; no continuous update schedule is promised. |
| `phishdestroy/destroylist` | Exact domain/URL reputation lookup for links pasted into emails | Dataset card says it syncs hourly from GitHub. The application reads the feed timestamp and warns if it appears stale. It is a domain blocklist, not an email corpus. |
| ThreatCluster REST API | Live incident/entity/dark-web search at query time | Separate optional API source. Requires `THREATCLUSTER_API_KEY`; current free tier documentation states 100 credits/day and a seven-day lookback. API results are labelled [L#], not treated as snapshot datasets. |
| `guychuk/cyber-threat-intelligence-reports` | Historical CTI reports, including older campaigns such as MiniDuke | Retained as an archive, not treated as current threat intelligence. Verify its dataset provenance and rights before redistribution. |

**Important freshness distinction:** Hugging Face and Kaggle datasets are usually published snapshots. A frequently refreshed feed can be downloaded again, but a static snapshot does not become live simply because it is hosted on Hugging Face. This notebook refreshes the PhishDestroy file when the notebook starts; the ThreatCluster snapshots remain dated snapshots until their publisher releases a new revision. When enabled and configured, the ThreatCluster API adds a separately cited live search result at question time. It does not convert the stored snapshots into live feeds.

**Assessment honesty:** This system reports the dates and limitations available in the source data. It must not invent financial losses, claim an unverified ransomware listing is a confirmed breach, or describe a feed as current when its timestamp is stale.

# Design Decisions and Rationale

## 1. Why these components?

- **Embedding model — `sentence-transformers/all-MiniLM-L6-v2`:** A compact sentence-transformer that can create semantic representations efficiently in a Colab environment. It is a reasonable baseline for semantic retrieval, but its retrieval quality on CTI terminology must be measured rather than assumed.
- **Vector store — ChromaDB:** Stores chunk embeddings and supports similarity search. The notebook uses Chroma semantic retrieval plus BM25 keyword retrieval and fuses the ranked results. It returns up to 3 chunks (`top_k=3`); this value must be evaluated, not assumed optimal.
- **Prompt template:** Constrains the model to answer from retrieved CTI evidence, disclose when evidence is insufficient, and cite retrieved source IDs.

## 2. Why use RAG?

RAG is a suitable approach because the application must find specific facts in a document collection at query time. The corpus can be updated without retraining the base model. RAG is not the only possible solution, but it is a good fit for this document-question-answering use case.

## 3. Decoding and factuality

For the revised generation configuration, `do_sample=False` uses greedy decoding, reducing variation between repeated runs. Temperature is not used when sampling is disabled. Deterministic decoding does **not** guarantee factual answers; source-grounding, retrieval quality, and evaluation are still required.

## 4. Target users

- SOC analysts who need to search historical and periodically refreshed CTI records.
- Threat hunters and incident responders researching reported threat actors, campaigns, tools, and techniques.
- Cybersecurity students and researchers studying historical threat intelligence.

**Important limitation:** The application supports research and triage by retrieving historical report evidence. It does not independently detect malware, detonate files, validate live indicators, or replace analyst judgement.

## Scope boundary and output requirements

The application keeps the existing Gemma 2 2B model and RAG approach. It adds newer, task-specific sources rather than changing the language model.

The assistant can:
- Answer questions from historical reports and curated CTI records.
- Retrieve incident summaries and report dates.
- Look up CVEs, severity, affected products, KEV status and known exploit fields.
- Look up ransomware leak-site claims by group, organisation, sector or country.
- Check URLs/domains in a pasted email against the downloaded PhishDestroy blocklist.
- Explain observed email warning signs and provide safe analyst checks.

The assistant cannot prove that an email is malicious from text alone, execute or safely detonate files, establish a confirmed breach from a ransomware group's claim, or calculate a calibrated probability from vector similarity. If financial loss is not present in a source, it must say **not reported in the retrieved data**.

For a breach/incident question, the answer should distinguish: (1) what happened / the issue, (2) target and likely objective, (3) affected systems or data, (4) reported impact or loss, and (5) documented or recommended mitigation. Unknown fields must remain unknown rather than being guessed.

## Strict approved-source grounding rule
Every CTI claim, incident detail, loss figure, threat actor attribution, and mitigation action must be supported by a retrieved record from the selected datasets or the separately labelled live ThreatCluster API channel. The assistant must not fill gaps from Gemma's pretrained knowledge. If approved sources do not contain a fact, it must say: **"Not found in the selected datasets or live API results."** This applies field-by-field, not only when retrieval returns zero documents. The user's submitted email may be described as direct input evidence, but it must not be presented as a verified CTI finding.


## Question-intent routing and traceable citations

The CTI interface now distinguishes definition questions, mitigation questions, direct factual questions, and explicit case-study requests. A definition or mitigation question (for example, “What is a wiper attack and how can it be mitigated?”) must not be reformatted as two incident case studies. Case-study retrieval is used only when the question explicitly requests cases/examples or asks a broad sector-plus-attack question.

Generated factual statements are instructed to cite evidence labels such as `[S1]`, `[C1]`, `[V1]`, `[R1]`, and `[L1]`. The application appends dataset cards for snapshot evidence and the API reference/fetch time for live evidence. The evidence panel shows record title, source type, date, record ID, excerpt, and available links. If approved records do not support a requested detail, the answer must say **Not found in the selected datasets or live API results.**


**Live API configuration:** Add `THREATCLUSTER_API_KEY` to Colab Secrets (or an environment variable) to enable optional live ThreatCluster search. The UI toggle controls whether a query uses it. For email triage, the default is off; when enabled, only an extracted domain is sent, never the email text.


# Import packages

##1. Environment Setup (Code Cell)
Install the required libraries for Hugging Face, LangChain, and vector embeddings to build your implementation evidence.

In [1]:
# Install compatible dependencies for Colab.
# If core packages are changed, restart the runtime before continuing.
%pip install -q bitsandbytes chromadb "gradio>=6,<7" datasets sentence-transformers transformers accelerate langchain langchain-community langchain-core langchain-huggingface langchain-text-splitters rank-bm25 fastapi uvicorn requests pandas numpy "opentelemetry-api>=1.39,<=1.42.1" "opentelemetry-sdk>=1.39,<=1.42.1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


## 1. Load the datasets

The notebook downloads public Hugging Face datasets. Several datasets are snapshots and may take time to download. The Gemma model may require accepting its access terms and providing a Hugging Face token; that authentication remains in the model-loading cell below.

## Data architecture note

Narrative sources (incident summaries, curated CTI knowledge and historical reports) are embedded for Chroma/BM25 retrieval. CVE and ransomware victim tables are queried as structured records only when relevant, so the notebook avoids embedding every row of the 60K-CVE and 20K-victim tables. The phishing-domain feed is also checked by exact domain matching rather than semantic similarity.

This separation improves traceability and reduces unnecessary embedding work. It does not guarantee a 15-second response: the local Gemma generation time is usually the dominant per-query cost. Check the retrieval and generation timings shown by the app.

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import json
import re
from collections import defaultdict

print("Loading ThreatCluster incident summaries...")
incident_ds = load_dataset("threatcluster/threat-incident-clusters", split="train")

print("Loading CVE exploitation records...")
cve_ds = load_dataset("threatcluster/cve-exploitation-signals", split="train")
cve_df = cve_ds.to_pandas()

print("Loading ransomware leak-site victim claims...")
ransomware_ds = load_dataset("threatcluster/ransomware-leak-site-victims", split="train")
ransomware_df = ransomware_ds.to_pandas()

print("Loading curated CTI Q&A knowledge...")
curated_cti_ds = load_dataset("reloading0101/threat-intelligence-dataset", split="train")

print("Loading historical CTI report archive...")
historical_ds = load_dataset("guychuk/cyber-threat-intelligence-reports", split="train")

print("\nDataset sizes")
print("Incident summaries:", len(incident_ds))
print("CVE records:", len(cve_df))
print("Ransomware listing claims:", len(ransomware_df))
print("Curated CTI Q&A:", len(curated_cti_ds))
print("Historical reports:", len(historical_ds))

def _as_search_text(value):
    if value is None:
        return ""
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple, dict, set)):
        if isinstance(value, set):
            value = sorted(value, key=str)
        return json.dumps(value, ensure_ascii=False, default=str)
    if pd.api.types.is_scalar(value) and pd.isna(value):
        return ""
    return str(value)

cve_search_columns = [c for c in ["cve_id", "description", "affected_vendors", "affected_products", "cwe_ids"] if c in cve_df.columns]
ransomware_search_columns = [c for c in ["group_name", "victim_name", "country", "sector"] if c in ransomware_df.columns]

cve_df["_search_blob"] = cve_df[cve_search_columns].apply(
    lambda row: " ".join(_as_search_text(v) for v in row.values).lower(), axis=1
)

if "cve_id" in cve_df.columns:
    cve_df["_cve_id_norm"] = cve_df["cve_id"].fillna("").astype(str).str.strip().str.upper()
    CVE_RECORDS_BY_ID = {
        str(cve_id): group.to_dict("records")
        for cve_id, group in cve_df[cve_df["_cve_id_norm"] != ""].groupby("_cve_id_norm", sort=False)
    }
else:
    cve_df["_cve_id_norm"] = ""
    CVE_RECORDS_BY_ID = {}

ransomware_df["_search_blob"] = ransomware_df[ransomware_search_columns].apply(
    lambda row: " ".join(_as_search_text(v) for v in row.values).lower(), axis=1
)

print("Structured CVE and ransomware lookup indexes are ready.")

Loading ThreatCluster incident summaries...


README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data.jsonl: reconstructing file:   0%|          |  0.00B / 25.9MB            

data.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19205 [00:00<?, ? examples/s]

Loading CVE exploitation records...


... (18 outputs left out)

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.1MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/978 [00:00<?, ? examples/s]


Dataset sizes
Incident summaries: 19205
CVE records: 60879
Ransomware listing claims: 20627
Curated CTI Q&A: 11042
Historical reports: 978
Structured CVE and ransomware lookup indexes are ready.


# Data Chunking (Text Splitting)

Narrative incident summaries, curated CTI knowledge and historical reports are split into overlapping chunks. CVE and ransomware-victim records are kept in structured tables and queried directly when the question is relevant. This avoids embedding every row of those large tables.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import requests
import json
import re
from datetime import datetime, timezone

# Convert any dataset field to readable, bounded text.
def as_text(value, limit=1800):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (dict, list, tuple, set)):
        if isinstance(value, set):
            value = sorted(value, key=str)
        value = json.dumps(value, ensure_ascii=False, default=str)
    return str(value)[:limit]

# Build a compact, in-memory lookup table for named incident case studies.
# This is created once at startup; it is NOT rebuilt for each question.
incident_df = incident_ds.to_pandas()
incident_case_columns = [
    c for c in [
        "title", "summary", "entities", "sources", "first_reported",
        "last_reported", "url", "cluster_id"
    ] if c in incident_df.columns
]
if incident_case_columns:
    incident_df["_case_search_blob"] = incident_df[incident_case_columns].apply(
        lambda row: " ".join(as_text(value, 2500) for value in row.values).lower(),
        axis=1
    )
else:
    incident_df["_case_search_blob"] = ""
print(f"Incident case lookup prepared: {len(incident_df):,} records.")

base_documents = []

# 1) Recent incident-cluster summaries. Titles/summaries are generated by the
# publisher and must be corroborated through their listed outlets when possible.
for row in incident_ds:
    title = as_text(row.get("title"), 300)
    summary = as_text(row.get("summary"), 1600)
    if not (title or summary):
        continue
    entities = as_text(row.get("entities"), 1200)
    outlets = as_text(row.get("sources"), 900)
    content = (
        f"Incident title: {title}\n"
        f"Incident summary: {summary}\n"
        f"Entities: {entities}\n"
        f"Reporting outlets: {outlets}\n"
        f"First reported: {as_text(row.get('first_reported'))}\n"
        f"Last reported: {as_text(row.get('last_reported'))}\n"
        f"Urgency: {as_text(row.get('urgency_level'))}\n"
        f"Threat score (publisher ranking, not a probability): {as_text(row.get('threat_score'))}\n"
        f"Severity score (publisher score): {as_text(row.get('severity_score'))}"
    )
    metadata = {
        "source_dataset": "threatcluster/threat-incident-clusters",
        "record_type": "incident_summary",
        "title": title or "Incident summary",
        "source_url": as_text(row.get("url"), 500),
        "record_date": as_text(row.get("last_reported") or row.get("first_reported"), 50),
        "record_id": as_text(row.get("cluster_id"), 100),
    }
    base_documents.append(Document(page_content=content, metadata=metadata))

# 2) Curated CTI instruction/answer records. These are secondary knowledge records,
# not primary news articles or original incident reports.
for row in curated_cti_ds:
    instruction = as_text(row.get("instruction"), 500)
    user_input = as_text(row.get("input"), 1000)
    output = as_text(row.get("output"), 2600)
    if not output:
        continue
    meta_obj = row.get("metadata") or {}
    if not isinstance(meta_obj, dict):
        meta_obj = {}
    category = as_text(meta_obj.get("category"), 100)
    grounding = as_text(meta_obj.get("grounding"), 250)
    source_refs = as_text(meta_obj.get("source_refs"), 700)
    content = (
        f"Curated CTI task: {instruction}\n"
        f"Input/entity: {user_input}\n"
        f"Category: {category}\n"
        f"Grounding sources: {grounding}\n"
        f"Source references: {source_refs}\n"
        f"Curated CTI answer: {output}"
    )
    metadata = {
        "source_dataset": "reloading0101/threat-intelligence-dataset",
        "record_type": "curated_cti_qa",
        "title": category or "Curated CTI knowledge",
        "source_url": "https://huggingface.co/datasets/reloading0101/threat-intelligence-dataset",
        "record_id": as_text(meta_obj.get("id"), 100),
    }
    base_documents.append(Document(page_content=content, metadata=metadata))

# 3) Historical archive. Retained for older campaigns and historical questions.
for row in historical_ds:
    report_text = as_text(row.get("text"), 100000)
    if not report_text.strip():
        continue
    report_url_match = re.search(r"Report_URL\s*:\s*(https?://\S+)", report_text)
    metadata = {
        "source_dataset": "guychuk/cyber-threat-intelligence-reports",
        "record_type": "historical_report",
        "title": as_text(row.get("title"), 300) or "Historical CTI report",
        "path": as_text(row.get("path"), 300),
        "record_id": as_text(row.get("id"), 100),
        "start_line": as_text(row.get("start_line"), 30),
        "end_line": as_text(row.get("end_line"), 30),
    }
    if report_url_match:
        metadata["source_url"] = report_url_match.group(1).rstrip(".,)")
    base_documents.append(Document(page_content=report_text, metadata=metadata))

# 4) PhishDestroy feed. This is used for exact URL/domain IOC matching, NOT vector search.
# The dataset card says it syncs hourly; check the published timestamp instead of assuming.
IOC_DOMAINS = set()
IOC_FEED_LAST_UPDATED = "Timestamp unavailable"
IOC_FEED_FETCHED_AT = datetime.now(timezone.utc).isoformat()
IOC_FEED_STATUS = "Unavailable"

IOC_FEED_URL = "https://huggingface.co/datasets/phishdestroy/destroylist/resolve/main/domains.txt"
IOC_README_URL = "https://huggingface.co/datasets/phishdestroy/destroylist/resolve/main/README.md?download=true"

try:
    feed_response = requests.get(IOC_FEED_URL, timeout=25, headers={"User-Agent": "CTI-RAG-Student-Project/1.0"})
    feed_response.raise_for_status()
    for line in feed_response.text.splitlines():
        candidate = line.strip().lower()
        if not candidate or candidate.startswith("#"):
            continue
        candidate = re.sub(r"^https?://", "", candidate)
        candidate = candidate.split("/")[0].split(":")[0].strip(".")
        if candidate:
            IOC_DOMAINS.add(candidate)

    readme_response = requests.get(IOC_README_URL, timeout=15, headers={"User-Agent": "CTI-RAG-Student-Project/1.0"})
    if readme_response.ok:
        timestamp_match = re.search(r"Last updated:\s*([^\r\n]+)", readme_response.text, re.I)
        if timestamp_match:
            IOC_FEED_LAST_UPDATED = timestamp_match.group(1).strip()
    IOC_FEED_STATUS = "Loaded"
except Exception as feed_error:
    print("WARNING: PhishDestroy feed could not be downloaded:", repr(feed_error))
    print("Email URL reputation will be shown as NOT CHECKED; it will not be treated as safe.")

print(f"PhishDestroy feed status: {IOC_FEED_STATUS}; domains loaded: {len(IOC_DOMAINS):,}")
print("PhishDestroy published timestamp:", IOC_FEED_LAST_UPDATED)

# This feed claims hourly refreshes; flag a published timestamp older than 6 hours.
try:
    parsed_feed_time = datetime.strptime(IOC_FEED_LAST_UPDATED.replace(" UTC", ""), "%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
    feed_age_hours = (datetime.now(timezone.utc) - parsed_feed_time).total_seconds() / 3600
    if feed_age_hours > 6:
        print(f"WARNING: The published IOC snapshot is {feed_age_hours:.1f} hours old. Treat matches as a lead and verify independently.")
except Exception:
    print("WARNING: Could not parse the IOC feed timestamp. Verify feed freshness before relying on it.")

# 5) CVE and ransomware target rows remain structured tables; we look them up only
# when a query is about a vulnerability or ransomware targeting. This avoids embedding
# tens of thousands of rows unnecessarily.
print(f"\nBuilding vector-search collection from {len(base_documents):,} narrative/curated records...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=120,
    add_start_index=True,
    separators=["\n\n", "\n", ". ", " ", ""]
)
docs = text_splitter.split_documents(base_documents)

print(f"Loaded {len(base_documents):,} base records and created {len(docs):,} chunks.")
print("Example metadata:", docs[0].metadata if docs else "No chunks created.")

Incident case lookup prepared: 19,205 records.
PhishDestroy feed status: Loaded; domains loaded: 236,530
PhishDestroy published timestamp: 2026-11-24 13:30 UTC*

Building vector-search collection from 31,225 narrative/curated records...
Loaded 31,225 base records and created 94,894 chunks.
Example metadata: {'source_dataset': 'threatcluster/threat-incident-clusters', 'record_type': 'incident_summary', 'title': 'MikroTik RouterOS Vulnerabilities Under Active Exploitation', 'source_url': 'https://threatcluster.io/cluster/971184ec', 'record_date': '2026-09-06 00:00:00', 'record_id': '971184ec', 'start_index': 0}


# Create the Vector Database

The narrative CTI chunks are embedded with the same sentence-transformer model and stored in ChromaDB. At query time, Chroma semantic similarity is combined with BM25 keyword search. Exact CVE and ransomware-victim lookups are handled separately.

**Strict grounding requirement:** A retrieved record must directly match the question. The app uses a conservative evidence gate before generation, and Gemma is instructed to use only supplied selected-dataset records and separately labelled live API results. Missing details are labelled "Not found in the selected datasets or live API results."

### Retrieval and evidence-gate update
The application distinguishes a request for a named case study from a normal technical explanation or comparison. A missing row in the incident-cluster table does not automatically block generation if a relevant historical report or curated CTI record was retrieved. Every generated factual statement must still be supported by those retrieved records. If no relevant evidence is retrieved, the assistant abstains. The hybrid search retrieves a wider candidate set and expands common C2, encryption, obfuscation, MiniDuke and zero-day terminology before ranking.


In [4]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
from collections import defaultdict
import re
import numpy as np

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Generating embeddings and building the vector store...")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="cti_refreshable_multisource"
)

def tokenize_cti(text):
    return re.findall(r"(?u)[a-z0-9]+(?:[._-][a-z0-9]+)*", str(text).lower())

print("Building BM25 keyword index and cross-dataset CVE chunk index...")
bm25_corpus = [tokenize_cti(doc.page_content) for doc in docs]
bm25_index = BM25Okapi(bm25_corpus)

# Map every CVE-YYYY-NNNN mentioned in curated CTI Q&A, incident clusters, and historical reports
CVE_CHUNKS_BY_ID = defaultdict(list)
_CVE_RE = re.compile(r"\bCVE-\d{4}-\d{3,}\b", flags=re.I)
for idx, doc in enumerate(docs):
    found_cves = set(m.upper() for m in _CVE_RE.findall(doc.page_content or ""))
    for cid in found_cves:
        if len(CVE_CHUNKS_BY_ID[cid]) < 6:
            CVE_CHUNKS_BY_ID[cid].append(doc)

MINIDUKE_CHUNK_INDICES = [
    i for i, doc in enumerate(docs)
    if "miniduke" in str((doc.metadata or {}).get("title", "")).lower()
    or "miniduke" in (doc.page_content or "").lower()
]

TOP_K = 8
CANDIDATE_K = 30


def is_reference_noise(doc):
    text = doc.page_content or ""
    low = text.lower()
    first_lines = "\n".join(text.splitlines()[:6]).lower()
    if "appendix: parallel and previous research" in low:
        return True
    if re.search(r"(?im)^\s*(references|bibliography)\s*$", first_lines):
        return True
    url_count = len(re.findall(r"https?://", text, flags=re.I))
    year_entries = len(re.findall(r"(?m)^\s*(?:\[\d+\]|\d+[.)])?.{0,140}\b(?:19|20)\d{2}\b", text))
    return url_count >= 3 and year_entries >= 2


def expand_cti_query(query):
    return str(query or "").strip()[:1800]


def _doc_key(doc):
    meta = doc.metadata or {}
    return (
        str(meta.get("source_dataset", "")),
        str(meta.get("record_id", meta.get("id", ""))),
        str(meta.get("start_index", "")),
        (doc.page_content or "")[:100]
    )


def hybrid_retrieve(query, k=TOP_K, candidate_k=CANDIDATE_K):
    search_query = expand_cti_query(query)
    vector_docs = vectorstore.similarity_search(search_query, k=candidate_k)

    bm25_stopwords = {
        "what", "which", "when", "where", "who", "why", "how", "can", "could",
        "would", "should", "please", "explain", "compare", "describe", "give",
        "show", "list", "the", "and", "for", "from", "with", "about", "this",
        "that", "into", "their", "there", "are", "was", "were", "is",
        "does", "did", "not", "use", "used", "using", "similar", "example",
        "examples", "step", "steps", "detail", "detailed", "simple", "terms"
    }
    query_tokens = [
        token for token in dict.fromkeys(tokenize_cti(search_query))
        if token not in bm25_stopwords and len(token) > 1
    ][:32]
    bm25_scores = bm25_index.get_scores(query_tokens)

    if len(bm25_scores) > candidate_k:
        candidate_indices = np.argpartition(bm25_scores, -candidate_k)[-candidate_k:]
        lexical_indices = candidate_indices[np.argsort(bm25_scores[candidate_indices])[::-1]]
    else:
        lexical_indices = np.argsort(bm25_scores)[::-1]
    lexical_docs = [docs[int(i)] for i in lexical_indices if float(bm25_scores[int(i)]) > 0]

    # Exact CVE narrative chunks (e.g., from reloading0101/threat-intelligence-dataset)
    exact_cve_docs = []
    for cid in dict.fromkeys(m.upper() for m in _CVE_RE.findall(search_query)):
        exact_cve_docs.extend(CVE_CHUNKS_BY_ID.get(cid, []))

    title_docs = []
    if "miniduke" in search_query.lower() and MINIDUKE_CHUNK_INDICES:
        titled = sorted(
            MINIDUKE_CHUNK_INDICES,
            key=lambda i: float(bm25_scores[i]),
            reverse=True
        )
        title_docs = [docs[i] for i in titled[:candidate_k] if float(bm25_scores[i]) > 0]

    fused = {}
    for ranked_docs, weight in [
        (exact_cve_docs, 2.5),
        (vector_docs, 1.0),
        (lexical_docs, 1.35),
        (title_docs, 1.5),
    ]:
        for rank, doc in enumerate(ranked_docs, start=1):
            if is_reference_noise(doc):
                continue
            key = _doc_key(doc)
            if key not in fused:
                fused[key] = {"doc": doc, "score": 0.0}
            fused[key]["score"] += weight / (60 + rank)

    ordered = sorted(fused.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ordered[:k]]

print(f"Hybrid retrieval ready: Chroma + BM25 + CVE chunk index ({len(CVE_CHUNKS_BY_ID):,} CVEs); top_k={TOP_K}.")

/tmp/ipykernel_418/680532252.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings and building the vector store...
Building BM25 keyword index and cross-dataset CVE chunk index...
Hybrid retrieval ready: Chroma + BM25 + CVE chunk index (7,205 CVEs); top_k=8.


# Authenticate and Load the LLM (Gemma 2 2B)

To use Gemma, must pass the Hugging Face token.

In [5]:
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)
from langchain_huggingface import HuggingFacePipeline

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            hf_token = None
if not hf_token:
    raise ValueError(
        "HF_TOKEN is missing. In Colab, add HF_TOKEN to Colab Secrets. "
        "In Hugging Face Spaces, add HF_TOKEN under Space Settings > Variables and secrets."
    )

os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
model_id = "google/gemma-2-2b-it"

if not torch.cuda.is_available():
    raise RuntimeError(
        "This 4-bit bitsandbytes configuration requires a compatible CUDA GPU. "
        "In Colab, select Runtime > Change runtime type > GPU."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",
    quantization_config=quantization_config
)
model.eval()
model.config.use_cache = True
if getattr(model, "generation_config", None) is not None:
    model.generation_config.use_cache = True

# 512 tokens allows multi-part CTI answers (definition + case + mitigations)
# and 7-section email triage outputs to finish all inline citations cleanly.
MAX_NEW_TOKENS = 512

generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,
    num_beams=1,
    use_cache=True,
    return_full_text=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=generation_pipeline)
print("Gemma 2 2B Instruct loaded successfully in 4-bit mode.")
print("GPU:", torch.cuda.get_device_name(0))
print(f"Generation cap: {MAX_NEW_TOKENS} new tokens.")

print("Warming up the model...")
with torch.inference_mode():
    _ = generation_pipeline(
        "<start_of_turn>user\nReply OK.<end_of_turn>\n<start_of_turn>model\n",
        max_new_tokens=2
    )
torch.cuda.synchronize()
print("Model warm-up complete.")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Gemma 2 2B Instruct loaded successfully in 4-bit mode.
GPU: Tesla T4
Generation cap: 512 new tokens.
Warming up the model...
Model warm-up complete.


# Construct the RAG Prompt Template

A highly disciplined prompt is necessary to control hallucination and ensure the model only uses the retrieved chunks.

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re
import numpy as np
import pandas as pd
from urllib.parse import urlparse


DATASET_CATALOG = {
    "threatcluster/threat-incident-clusters": {"owner": "ThreatCluster", "year": "2026", "title": "threat-incident-clusters", "url": "https://huggingface.co/datasets/threatcluster/threat-incident-clusters", "note": "Snapshot generated 2026-09-06; summaries are publisher-generated clusters.", "license": "CC BY 4.0"},
    "threatcluster/cve-exploitation-signals": {"owner": "ThreatCluster", "year": "2026", "title": "cve-exploitation-signals", "url": "https://huggingface.co/datasets/threatcluster/cve-exploitation-signals", "note": "Snapshot generated 2026-09-06.", "license": "CC BY 4.0"},
    "threatcluster/ransomware-leak-site-victims": {"owner": "ThreatCluster", "year": "2026", "title": "ransomware-leak-site-victims", "url": "https://huggingface.co/datasets/threatcluster/ransomware-leak-site-victims", "note": "Snapshot generated 2026-09-06; listings are group claims, not independently verified breaches.", "license": "CC BY 4.0"},
    "reloading0101/threat-intelligence-dataset": {"owner": "reloading0101", "year": "n.d.", "title": "threat-intelligence-dataset", "url": "https://huggingface.co/datasets/reloading0101/threat-intelligence-dataset", "note": "Curated CTI question/answer records; secondary knowledge source.", "license": "CC BY-NC-SA 4.0"},
    "guychuk/cyber-threat-intelligence-reports": {"owner": "guychuk", "year": "n.d.", "title": "cyber-threat-intelligence-reports", "url": "https://huggingface.co/datasets/guychuk/cyber-threat-intelligence-reports", "note": "Historical report archive; not a live feed.", "license": "See dataset card"},
    "phishdestroy/destroylist": {"owner": "phishdestroy", "year": "n.d.", "title": "destroylist", "url": "https://huggingface.co/datasets/phishdestroy/destroylist", "note": "Phishing/scam domain feed; freshness must be checked using its published timestamp.", "license": "See dataset card"},
    "nist/nvd": {"owner": "NIST", "year": "n.d.", "title": "National Vulnerability Database (live API)", "url": "https://nvd.nist.gov/", "note": "Authoritative CVE record fetched live from the NVD 2.0 API at request time.", "license": "U.S. government work"}
}


def _dataset_info(dataset_name):
    return DATASET_CATALOG.get(dataset_name, {
        "owner": "Unknown publisher", "year": "n.d.",
        "title": dataset_name or "Unknown dataset", "url": "", "note": ""
    })


def format_docs(retrieved_docs, max_chars_per_doc=850, label_prefix="S"):
    formatted = []
    for index, doc in enumerate(retrieved_docs, start=1):
        meta = doc.metadata or {}
        dataset_name = meta.get("source_dataset", "Unknown dataset")
        info = _dataset_info(dataset_name)
        label = f"{label_prefix}{index}"
        title = meta.get("title", "Untitled CTI record")
        record_type = meta.get("record_type", "CTI record")
        date = meta.get("record_date", "Date not supplied")
        record_id = meta.get("record_id", "Not supplied")
        source_url = meta.get("source_url", "")
        if source_url == info.get("url"):
            source_url = ""
        evidence = (doc.page_content or "")[:max_chars_per_doc]
        formatted.append(
            f"[{label}]\n"
            f"Title: {title}\n"
            f"Record type: {record_type}\n"
            f"Dataset: {dataset_name}\n"
            f"Dataset card: {info['url'] or 'Not supplied'}\n"
            f"Record date: {date}\n"
            f"Record ID: {record_id}\n"
            f"Original record/source URL: {source_url or 'Not supplied'}\n"
            f"Evidence: {evidence}"
        )
    return "\n\n".join(formatted)


_STOP_WORDS = {
    "what", "which", "when", "where", "from", "with", "about", "does", "show",
    "reports", "report", "data", "dataset", "attack", "attacks", "incident", "incidents",
    "security", "cyber", "threat", "target", "targets", "impact", "loss", "losses",
    "explain", "please", "tell", "give", "latest", "current", "recent", "find", "known",
    "documented", "cases", "case", "two", "provide", "real", "world", "example", "examples",
    "step", "simple", "terms", "list"
}


def _query_terms(query, max_terms=10):
    terms = re.findall(r"[a-z0-9][a-z0-9._-]{2,}", str(query).lower())
    return [t for t in dict.fromkeys(terms) if t not in _STOP_WORDS][:max_terms]


def _rank_rows(frame, terms, max_rows=3):
    if frame.empty or not terms:
        return frame.iloc[0:0]
    clean_terms = list(dict.fromkeys(str(t).lower() for t in terms if len(str(t)) >= 3))
    if not clean_terms:
        return frame.iloc[0:0]
    blob_col = "_case_search_blob" if "_case_search_blob" in frame.columns else "_search_blob"
    alternatives = "|".join(re.escape(t) for t in sorted(clean_terms, key=len, reverse=True))
    pattern = rf"(?<![a-z0-9])(?:{alternatives})(?![a-z0-9])"
    matches = frame[blob_col].str.findall(pattern, flags=re.I)
    scores = matches.str.len().fillna(0).astype("int32")
    matched = frame.loc[scores > 0].copy()
    if matched.empty:
        return matched
    matched["_match_score"] = scores.loc[matched.index]
    sort_cols = ["_match_score"]
    ascending = [False]
    for date_col in ("published_date", "last_reported", "discovered"):
        if date_col in matched.columns:
            sort_cols.append(date_col)
            ascending.append(False)
            break
    return matched.sort_values(sort_cols, ascending=ascending).head(max_rows)


_SEMANTIC_INTENT_CACHE = {}
_SEMANTIC_INTENT_PROTOTYPES = [
    "Find and describe documented real-world cyber incidents affecting organisations.",
    "Give named incident case studies and explain what happened, the impact, and the response.",
    "Provide examples of organisations that experienced a cyberattack, with evidence from reports.",
    "Explain a cybersecurity concept or define a vulnerability.",
    "Look up structured metadata for a specific CVE identifier.",
    "Explain technical mitigation steps or defensive recommendations."
]
_SEMANTIC_INTENT_MATRIX = None

def _get_semantic_intent_matrix():
    global _SEMANTIC_INTENT_MATRIX
    if _SEMANTIC_INTENT_MATRIX is None:
        values = np.asarray(
            embeddings.embed_documents(_SEMANTIC_INTENT_PROTOTYPES), dtype=np.float32
        )
        values /= np.maximum(np.linalg.norm(values, axis=1, keepdims=True), 1e-12)
        _SEMANTIC_INTENT_MATRIX = values
    return _SEMANTIC_INTENT_MATRIX


def semantic_case_request(query):
    cache_key = str(query or "").strip()
    if cache_key in _SEMANTIC_INTENT_CACHE:
        return _SEMANTIC_INTENT_CACHE[cache_key]
    low = cache_key.lower()
    if re.search(r"\b(case stud(?:y|ies)|real[- ]world (?:case|example)|documented .{0,35} cases?)\b", low):
        _SEMANTIC_INTENT_CACHE[cache_key] = True
        return True
    try:
        qv = np.asarray(embeddings.embed_query(cache_key), dtype=np.float32)
        qv /= max(float(np.linalg.norm(qv)), 1e-12)
        scores = _get_semantic_intent_matrix() @ qv
        case_score = float(np.max(scores[:3]))
        other_score = float(np.max(scores[3:]))
        result = case_score >= 0.30 and case_score > other_score + 0.015
        _SEMANTIC_INTENT_CACHE[cache_key] = result
        return result
    except Exception:
        return False


def retrieve_incident_case_studies(query, max_cases=2):
    """Retrieve named incident summaries using both lexical table ranking and Chroma semantic search."""
    if incident_df.empty:
        return []

    candidate_rows = []
    # 1) Lexical match on incident_df (ensures sector + attack terms like healthcare + ransomware hit)
    terms = _query_terms(query, max_terms=8)
    if terms:
        lex_df = _rank_rows(incident_df, terms, max_rows=max_cases * 3)
        for _, row in lex_df.iterrows():
            candidate_rows.append(row)

    # 2) Semantic match from Chroma incident-clusters
    try:
        matched_docs = vectorstore.similarity_search(
            str(query), k=max(16, max_cases * 8),
            filter={"source_dataset": "threatcluster/threat-incident-clusters"}
        )
    except Exception:
        matched_docs = [
            doc for doc in vectorstore.similarity_search(str(query), k=40)
            if (doc.metadata or {}).get("source_dataset") == "threatcluster/threat-incident-clusters"
        ]

    global _INCIDENT_ROWS_BY_ID
    try:
        rows_by_id = _INCIDENT_ROWS_BY_ID
    except NameError:
        rows_by_id = {}
        if "cluster_id" in incident_df.columns:
            for _, row in incident_df.iterrows():
                rows_by_id[as_text(row.get("cluster_id"), 100)] = row
        _INCIDENT_ROWS_BY_ID = rows_by_id

    for matched in matched_docs:
        meta = matched.metadata or {}
        cid = as_text(meta.get("record_id"), 100)
        if cid in rows_by_id:
            candidate_rows.append(rows_by_id[cid])

    case_docs, seen = [], set()
    for row in candidate_rows:
        cid = as_text(row.get("cluster_id"), 100)
        title = as_text(row.get("title"), 350) or "Untitled incident"
        dedupe_key = cid or title.lower()
        if dedupe_key in seen:
            continue
        seen.add(dedupe_key)

        summary = as_text(row.get("summary"), 2200)
        entities = as_text(row.get("entities"), 1000)
        outlets = as_text(row.get("sources"), 900)
        first_reported = as_text(row.get("first_reported"), 50)
        last_reported = as_text(row.get("last_reported"), 50)
        source_url = as_text(row.get("url"), 600)

        content = (
            f"Named incident case: {title}\nSummary: {summary}\n"
            f"Entities extracted by dataset publisher: {entities}\n"
            f"Reporting outlets: {outlets}\nFirst reported: {first_reported}\n"
            f"Last reported: {last_reported}\n"
            "Evidence caveat: This is a publisher-generated incident-cluster summary. "
            "Do not infer financial loss, patient-care impact, or response actions unless explicitly stated."
        )
        case_docs.append(Document(
            page_content=content,
            metadata={
                "source_dataset": "threatcluster/threat-incident-clusters",
                "record_type": "incident_case_study",
                "title": title,
                "source_url": source_url,
                "record_date": last_reported or first_reported,
                "record_id": cid,
            }
        ))
        if len(case_docs) >= max_cases:
            break
    return case_docs


def retrieve_exact_cve_rows(query):
    requested = list(dict.fromkeys(
        value.strip().upper()
        for value in re.findall(r"\bCVE-\d{4}-\d{3,}\b", str(query or ""), flags=re.I)
    ))
    rows = []
    for cve_id in requested:
        rows.extend(CVE_RECORDS_BY_ID.get(cve_id, []))
    return requested, rows


def retrieve_structured_context(query):
    q = str(query)
    low = q.lower()
    sections = []

    cve_ids, exact_records = retrieve_exact_cve_rows(q)
    cve_rows = pd.DataFrame(exact_records) if exact_records else cve_df.iloc[0:0]
    if cve_ids and not cve_rows.empty:
        dedupe_column = "_cve_id_norm" if "_cve_id_norm" in cve_rows.columns else "cve_id"
        cve_rows = cve_rows.drop_duplicates(subset=[dedupe_column])
    elif cve_ids:
        cve_rows = cve_df.iloc[0:0]
    elif re.search(r"\b(cve|vulnerab\w*|exploit\w*|patch\w*|kev|affected products?)\b", low):
        cve_rows = _rank_rows(cve_df, _query_terms(q), max_rows=3)

    for i, (_, row) in enumerate(cve_rows.iterrows(), start=1):
        sections.append(
            f"[V{i}]\n"
            f"CVE: {as_text(row.get('cve_id'))}\n"
            f"Description: {as_text(row.get('description'), 900)}\n"
            f"Published date: {as_text(row.get('published_date'))}\n"
            f"CVSS v3: {as_text(row.get('cvss_v3_score'))} ({as_text(row.get('cvss_v3_severity'))})\n"
            f"Affected vendors: {as_text(row.get('affected_vendors'), 500)}\n"
            f"Affected products: {as_text(row.get('affected_products'), 500)}\n"
            f"CISA KEV listed in this snapshot: {as_text(row.get('in_kev'))}\n"
            f"KEV added / due dates: {as_text(row.get('kev_added_date'))} / {as_text(row.get('kev_due_date'))}\n"
            f"Public exploit known / count: {as_text(row.get('has_exploit'))} / {as_text(row.get('exploit_count'))}\n"
            f"Ransomware use field: {as_text(row.get('ransomware_use'))}\n"
            "Dataset: threatcluster/cve-exploitation-signals. "
            "Dataset card: https://huggingface.co/datasets/threatcluster/cve-exploitation-signals. "
            "Snapshot generated 2026-09-06."
        )

    if re.search(r"\b(leak[\s-]?site|victim (?:listings?|claims?)|ransomware listings?|which organisations were listed)\b", low):
        victim_rows = _rank_rows(ransomware_df, _query_terms(q), max_rows=3)
        for i, (_, row) in enumerate(victim_rows.iterrows(), start=1):
            sections.append(
                f"[R{i}]\n"
                f"Group claim: {as_text(row.get('group_name'))}\n"
                f"Named organisation: {as_text(row.get('victim_name'))}\n"
                f"Sector / country: {as_text(row.get('sector'))} / {as_text(row.get('country'))}\n"
                f"First observed / published: {as_text(row.get('discovered'))} / {as_text(row.get('published'))}\n"
                f"Delisted: {as_text(row.get('delisted'))}\n"
                "Dataset: threatcluster/ransomware-leak-site-victims. "
                "Dataset card: https://huggingface.co/datasets/threatcluster/ransomware-leak-site-victims. "
                "Important caveat: this is a ransomware-group leak-site claim, not independent proof of a confirmed breach, data loss or payment. "
                "The dataset does not contain a verified financial-loss amount. Snapshot generated 2026-09-06."
            )

    return "\n\n".join(sections) if sections else "No matching structured CVE or ransomware-target records were selected for this query."


def _refang_text(text):
    t = str(text or "")
    t = re.sub(r"\bhxxps://", "https://", t, flags=re.I)
    t = re.sub(r"\bhxxp://", "http://", t, flags=re.I)
    t = t.replace("[.]", ".").replace("(.)", ".")
    return t


def extract_email_urls(text):
    clean = _refang_text(text)
    return re.findall(r"(?:https?://|www\.)[^\s<>\]\[()\"']+", clean, flags=re.I)


def extract_email_sender_domains(text):
    clean = _refang_text(text)
    emails = re.findall(r"[A-Za-z0-9._%+-]+@([A-Za-z0-9.-]+\.[A-Za-z]{2,})", clean)
    return list(dict.fromkeys(d.lower().strip(".") for d in emails))


def _normalise_domain(url):
    value = _refang_text(url).strip().rstrip(".,);]\"'")
    if value.lower().startswith("www."):
        value = "http://" + value
    elif not re.match(r"^https?://", value, flags=re.I):
        value = "http://" + value
    try:
        host = (urlparse(value).hostname or "").lower().strip(".")
        return host[4:] if host.startswith("www.") else host
    except Exception:
        return ""


def check_email_domains(email_text):
    """Exact/suffix domain matching for both embedded URLs and sender email domains."""
    urls = extract_email_urls(email_text)
    sender_domains = extract_email_sender_domains(email_text)
    candidates = []
    for url in urls:
        d = _normalise_domain(url)
        if d:
            candidates.append((url, d, "url"))
    for sd in sender_domains:
        candidates.append((f"@{sd}", sd, "sender_domain"))

    results, seen = [], set()
    for raw_item, domain, kind in candidates:
        if (domain, kind) in seen:
            continue
        seen.add((domain, kind))
        matches = []
        candidate = domain
        while candidate:
            if candidate in IOC_DOMAINS:
                matches.append(candidate)
            if "." not in candidate:
                break
            candidate = candidate.split(".", 1)[1]
        results.append({"url": raw_item, "domain": domain, "kind": kind, "matches": matches[:5]})
    return results


gemma_prompt_template = """<start_of_turn>user
You are a strict, evidence-bound Cyber Threat Intelligence analyst. You are NOT a general-knowledge answer bot.

SOURCE RULES:
1. Use ONLY the supplied selected-dataset records and the separately labelled live ThreatCluster API records. Never use pretrained knowledge, memory, assumptions or unlisted websites to fill gaps.
2. Every factual statement must end with a supplied evidence label: [S1], [S2], [C1], [C2], [V1], [R1], [L1], or [N1]. Never invent a citation label. [N#] labels are live NVD records; [L#] labels are live ThreatCluster API records.
3. Use a record only when it directly supports the point.
4. If a requested fact or sub-part is absent from all supplied approved sources, write: "Not found in the selected datasets or live API results." Do not guess.
5. Never invent an incident, actor, target, date, statistic, loss, impact, mitigation or citation. Do not merge different incidents.
6. A ransomware leak-site listing is a group's claim, not independent proof of a confirmed breach, data theft, payment or financial loss.
7. Incident-cluster summaries are publisher-generated summaries. Attribute claims to the relevant dataset or live API record.
8. Mitigation must be explicitly documented in the supplied records. Do not add generic advice from model memory. If absent, say "Not found in the selected datasets or live API results."
9. Do not output a generic statement that only selected datasets were used. The application appends source names and links after your answer.

QUESTION-SPECIFIC RESPONSE INSTRUCTION:
{response_instruction}

ANSWER QUALITY RULES:
- First sentence must directly answer the user's actual question.
- Answer EVERY sub-part requested by the user (e.g., definition, case study, and mitigations if all three are asked).
- Place the exact citation tag (e.g., [S1] or [C1]) at the end of EVERY factual sentence or bullet item.
- Keep the answer clear, well-structured, and strictly grounded in the evidence below.

EXAMPLE OF CORRECT FORMAT:
- A wiper is malware that destroys data on infected systems. [S1]
- The incident disrupted hospital appointment systems. [C1]
- Step 1: Apply the vendor patch documented in the record. [V1]

CASE STUDY RECORDS:
{case_study_context}

NARRATIVE RETRIEVED RECORDS:
{context}

STRUCTURED CVE / RANSOMWARE RECORDS:
{structured_context}

USER QUESTION:
{input}
<end_of_turn>
<start_of_turn>model
"""

prompt = PromptTemplate(
    template=gemma_prompt_template,
    input_variables=["context", "structured_context", "case_study_context", "input", "response_instruction"]
)
qa_chain = prompt | llm | StrOutputParser()


email_triage_template = """<start_of_turn>user
You are a strict, evidence-bound SOC email triage assistant.

SOURCE BOUNDARIES:
- Describe observable features literally present in the submitted email as observations from the submitted email, not CTI dataset facts.
- For historical campaigns, threat actors, malware families, known malicious domains, confirmed breaches, impacts, financial losses, and mitigation actions, use ONLY the supplied selected-dataset evidence ([S1], [S2]), live API evidence ([L1]), and the domain-feed result ([IOC1]).
- Do NOT use pretrained/general knowledge to fill missing CTI facts.
- If a requested CTI fact is absent from all approved sources, write: "Not found in the selected datasets or live API results."
- Preserve the preliminary rule-based label exactly: {rule_assessment}.
- Actual loss is not established by the email itself; only state a loss when an approved source record explicitly documents it.
- Mitigation/response actions: include only actions explicitly stated in supplied approved-source records ([S1], [S2]). If absent, say "Not found in the selected datasets or live API results."

Return concise plain language using this exact structure:
## Email triage result
**Assessment:** {rule_assessment}
**Reason:** [1-2 sentences based only on observed email text, domain-feed results, and dataset matches.]

## Observed warning signs
[2-4 bullets visible in the submitted email.]

## Possible objective and target
- **Possible objective:** [only if supported by email wording or approved source evidence; otherwise exact not-found wording]
- **Target:** [only what the email identifies or an approved source directly supports; otherwise exact not-found wording]
- **Actual loss:** [state only if explicitly documented in an approved record; otherwise exact not-found wording]

## Domain-feed check
[Report exact match/no match for URLs and sender domains with [IOC1] and feed timestamp/status.]

## Related CTI evidence
[Only directly relevant selected-dataset [S#] or live API [L#] evidence. If none, write: "Not found in the selected datasets or live API results."]

## Documented mitigation / response
[Only what approved source records [S#] explicitly say. If none, write: "Not found in the selected datasets or live API results."]

## Evidence limitations
[Briefly distinguish email observations from dataset-supported intelligence.]

Observed indicators:
{indicators}

Domain-feed results:
{ioc_results}

Feed published timestamp: {feed_timestamp}
Feed status: {feed_status}

Submitted email:
{email_text}

Retrieved selected-dataset and live API CTI evidence:
{context}
<end_of_turn>
<start_of_turn>model
"""

email_triage_prompt = PromptTemplate(
    template=email_triage_template,
    input_variables=["indicators", "ioc_results", "feed_timestamp", "feed_status", "email_text", "context", "rule_assessment"]
)
email_triage_chain = email_triage_prompt | llm | StrOutputParser()


def detect_email_indicators(email_text: str):
    text = _refang_text(email_text or "")
    lower = text.lower()
    findings = []

    if re.search(r"\b(urgent|immediately|act now|final warning|within\s+\d+\s+hours?)\b|account.{0,40}(suspend|disable|lock|terminate|expire)", lower, re.I):
        findings.append("Pressure or account-threat language is present.")
    if re.search(r"(enter|provide|confirm|verify|submit|update|reset|share).{0,70}(password|passcode|credentials|one[- ]?time code|MFA code|login details)|(password|passcode|credentials|one[- ]?time code|MFA code|login details).{0,70}(enter|provide|confirm|verify|submit|update|reset|share)", lower, re.I | re.S):
        findings.append("The message appears to request credentials or authentication information.")
    urls = extract_email_urls(text)
    if urls:
        findings.append(f"The message contains {len(urls)} visible URL(s).")
    if re.search(r"\b(wire transfer|bank account|payment details|gift card|change.{0,25}bank details|invoice|overdue payment)\b", lower, re.I):
        findings.append("The message requests or discusses a payment, invoice, or banking action.")
    if re.search(r"\.(exe|scr|js|jse|vbs|vbe|hta|iso|img|lnk|bat|cmd|ps1|docm|xlsm|zip|rar)\b", lower, re.I):
        findings.append("A potentially executable, archive, or macro-enabled attachment is mentioned.")
    if re.search(r"password[- ]protected|enable macros|enable content|disable security|protected document", lower, re.I):
        findings.append("The message asks the recipient to bypass a security control or enable document content.")
    if not findings:
        findings.append("No matches were found by the built-in text checklist; this does not establish that the email is safe.")
    return findings


def classify_email_preliminary(email_text: str, findings):
    findings_text = " ".join(findings).lower()
    if "no matches were found" in findings_text:
        return "INSUFFICIENT EVIDENCE"
    credential = "request credentials" in findings_text
    payment = "payment, invoice, or banking" in findings_text
    pressure = "pressure or account-threat" in findings_text
    urls = "visible url" in findings_text
    attachment = "attachment is mentioned" in findings_text or "bypass a security control" in findings_text
    if (credential or payment) and (pressure or urls or attachment):
        return "POTENTIALLY SUSPICIOUS"
    return "NEEDS FURTHER REVIEW"

In [7]:
# ---------------------------------------------------------------------
# Optional live ThreatCluster API integration
# ---------------------------------------------------------------------
import os
import time
import requests
from datetime import datetime, timezone
from functools import lru_cache
from langchain_core.documents import Document

THREATCLUSTER_API_BASE = "https://threatcluster.io/api/public/v1"
THREATCLUSTER_API_REFERENCE = "https://threatcluster.io/about/api-reference"
THREATCLUSTER_API_SEARCH_DOCS = "https://threatcluster.io/about/api-reference/threats"
_LIVE_API_CACHE = {}
_LIVE_API_CACHE_TTL_SECONDS = 600  # Cache repeated searches for 10 minutes to conserve credits.


def get_threatcluster_api_key():
    """Read API key from environment first, then Colab Secrets. Never print the key."""
    key = os.getenv("THREATCLUSTER_API_KEY")
    if key:
        return key.strip()
    try:
        from google.colab import userdata
        key = userdata.get("THREATCLUSTER_API_KEY")
        return key.strip() if key else None
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("THREATCLUSTER_API_KEY")
        return key.strip() if key else None
    except Exception:
        return None



def _live_search_term(query):
    """
    Compress the analyst's natural-language question into a compact search phrase.
    This does not decide whether live search is needed; the UI toggle does that.
    """
    text = re.sub(r"\s+", " ", str(query or "")).strip()
    cves = re.findall(r"\bCVE-\d{4}-\d{3,}\b", text, flags=re.I)
    if cves:
        return " ".join(dict.fromkeys(x.upper() for x in cves))[:120]
    stop = {
        "what", "which", "when", "where", "who", "why", "how", "does", "do",
        "is", "are", "was", "were", "the", "a", "an", "and", "or", "of", "to",
        "for", "in", "on", "at", "by", "with", "from", "about", "please",
        "give", "show", "find", "tell", "explain", "describe", "could", "would",
        "should", "me", "their", "its", "they", "them"
    }
    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9._-]{2,}", text)
    kept, seen = [], set()
    for token in tokens:
        key = token.lower()
        if key not in stop and key not in seen:
            kept.append(token)
            seen.add(key)
    return " ".join(kept[:9])[:160]


def _live_api_document(item, bucket, fetched_at, request_cost, credits_left):
    """Turn one API hit into a traceable document. Preserve only fields actually returned."""
    item = item if isinstance(item, dict) else {"value": item}
    title = str(item.get("title") or item.get("name") or item.get("victim_name") or item.get("entity_value") or f"ThreatCluster {bucket} record")[:300]
    summary = item.get("summary") or item.get("description") or item.get("snippet") or ""
    entities = item.get("entities") or item.get("entity_names") or []
    if isinstance(entities, (dict, list)):
        entities_text = json.dumps(entities, ensure_ascii=False, default=str)[:1000]
    else:
        entities_text = str(entities)[:1000]
    date_value = (
        item.get("last_reported") or item.get("published_at") or item.get("first_reported")
        or item.get("created_at") or item.get("discovered_at") or item.get("date") or "Date not supplied"
    )
    record_id = str(item.get("short_id") or item.get("cluster_id") or item.get("id") or item.get("slug") or "Not supplied")[:120]
    score = item.get("threat_score")
    urgency = item.get("urgency") or item.get("urgency_level")
    source_links = item.get("sources") or item.get("articles") or []
    source_urls = []
    if isinstance(source_links, list):
        for source in source_links[:4]:
            if isinstance(source, dict):
                u = source.get("url") or source.get("source_url") or source.get("link")
                if u and str(u).startswith("http"):
                    source_urls.append(str(u))
            elif isinstance(source, str) and source.startswith("http"):
                source_urls.append(source)
    direct_url = item.get("url") or item.get("source_url") or item.get("primary_url")
    if direct_url and str(direct_url).startswith("http"):
        source_urls.insert(0, str(direct_url))
    fields = []
    if summary:
        fields.append(f"Summary: {str(summary)[:2600]}")
    if entities_text:
        fields.append(f"Entities returned by API: {entities_text}")
    if date_value:
        fields.append(f"Reported/record date returned by API: {date_value}")
    if score is not None:
        fields.append(f"Publisher threat score (not a probability): {score}")
    if urgency is not None:
        fields.append(f"Publisher urgency: {urgency}")
    # Include a compact field excerpt to retain bucket-specific information without dumping raw payloads.
    keep = {k: v for k, v in item.items() if k in {
        "entity_type", "entity_value", "category", "country", "sector", "group_name",
        "victim_name", "cve_id", "severity", "cvss_v3_score", "in_kev", "has_exploit",
        "threat_score", "urgency", "urgency_level", "first_reported", "last_reported"
    } and v not in (None, "", [], {})}
    if keep:
        fields.append("Additional returned fields: " + json.dumps(keep, ensure_ascii=False, default=str))
    if bucket == "darkweb":
        fields.append("Source-type caveat: this is a dark-web/leak-site record returned by the API; treat victim statements as claims unless independently corroborated.")
    if source_urls:
        fields.append("Original reporting/source URLs: " + " | ".join(dict.fromkeys(source_urls[:4])))
    fields.append(f"Live API fetched at (UTC): {fetched_at}")
    fields.append(f"API request cost: {request_cost if request_cost is not None else 'Not returned'}; credits remaining: {credits_left if credits_left is not None else 'Not returned'}")
    content = f"Live ThreatCluster API {bucket} record: {title}\n" + "\n".join(fields)
    return Document(
        page_content=content,
        metadata={
            "source_dataset": "threatcluster-live-api",
            "source_kind": "live_api",
            "record_type": f"live_{bucket}",
            "title": title,
            "record_id": record_id,
            "record_date": str(date_value),
            "source_url": source_urls[0] if source_urls else THREATCLUSTER_API_REFERENCE,
            "api_endpoint": "/search",
            "api_docs_url": THREATCLUSTER_API_SEARCH_DOCS,
            "fetched_at": fetched_at,
            "request_cost": str(request_cost) if request_cost is not None else "Not returned",
            "credits_remaining": str(credits_left) if credits_left is not None else "Not returned",
            "bucket": bucket,
        }
    )


def fetch_threatcluster_live(query, enabled=True, privacy_safe_term=None):
    """Search ThreatCluster live API with one keyword, 7-day window and a 10-minute cache."""
    if not enabled:
        return [], "Live API not requested for this query.", {}
    key = get_threatcluster_api_key()
    if not key:
        return [], "Live API not configured: add THREATCLUSTER_API_KEY to Colab Secrets or the Space secret settings.", {}
    search_term = str(privacy_safe_term or _live_search_term(query)).strip()
    if len(search_term) < 2:
        return [], "Live API skipped: no suitable two-character search term was extracted.", {}

    now = time.time()
    cache_key = search_term.lower()
    cached = _LIVE_API_CACHE.get(cache_key)
    if cached and (now - cached["cached_at"]) < _LIVE_API_CACHE_TTL_SECONDS:
        return cached["docs"], cached["status"] + " (cached; no new API credit used)", cached["meta"]

    headers = {"X-API-Key": key, "Accept": "application/json", "User-Agent": "CTI-RAG-Student-Project/1.0"}
    params = {"q": search_term, "limit": 5, "days": 7, "include_articles": "false"}
    try:
        response = requests.get(
            f"{THREATCLUSTER_API_BASE}/search", headers=headers, params=params, timeout=(5, 15)
        )
        request_cost = response.headers.get("X-Request-Cost")
        credits_left = response.headers.get("X-RateLimit-Remaining")
        response.raise_for_status()
        payload = response.json()
        fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
        docs_out = []
        for bucket in ("clusters", "entities", "darkweb"):
            records = payload.get(bucket, []) if isinstance(payload, dict) else []
            if isinstance(records, list):
                for item in records[:5]:
                    docs_out.append(_live_api_document(item, bucket, fetched_at, request_cost, credits_left))
        docs_out = docs_out[:8]
        status = (
            f"Live ThreatCluster API searched `{search_term}` (last 7 days); "
            f"{len(docs_out)} records returned; request cost {request_cost or 'not supplied'} credit(s); "
            f"credits remaining today {credits_left or 'not supplied'}; fetched {fetched_at} UTC."
        )
        meta = {"query": search_term, "fetched_at": fetched_at, "request_cost": request_cost, "credits_remaining": credits_left, "days": 7}
        _LIVE_API_CACHE[cache_key] = {"cached_at": now, "docs": docs_out, "status": status, "meta": meta}
        return docs_out, status, meta
    except requests.HTTPError as exc:
        status = f"Live API request failed ({response.status_code}): {str(exc)[:300]}"
        if response.status_code == 403:
            status += " Check the API key's read scope and the free-tier seven-day lookback."
        elif response.status_code == 429:
            status += " The API rate or credit limit may have been reached."
        return [], status, {"http_status": response.status_code}
    except (requests.RequestException, ValueError) as exc:
        return [], f"Live API unavailable: {type(exc).__name__}: {str(exc)[:250]}", {}


# ---------------------------------------------------------------------
# NVD live CVE fallback (NIST public REST API, no key required)
# Used when a requested CVE is absent from the local structured snapshot so
# that any CVE question can still be answered from an authoritative source.
# ---------------------------------------------------------------------
_NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
_NVD_CACHE = {}
_NVD_CACHE_TTL_SECONDS = 1800


def _nvd_affected_products(item):
    """Flatten NVD CPE configurations into readable 'Vendor Product version' strings."""
    affected = []
    for config in item.get("configurations") or []:
        for node in config.get("nodes") or []:
            for match in node.get("cpeMatch") or []:
                if not match.get("vulnerable"):
                    continue
                parts = str(match.get("criteria") or "").split(":")
                if len(parts) <= 5:
                    continue
                vendor = parts[3].replace("_", " ")
                product = parts[4].replace("_", " ")
                version = parts[5].replace("_", " ")
                rng = []
                if match.get("versionStartIncluding"):
                    rng.append(f"from {match['versionStartIncluding']}")
                if match.get("versionEndExcluding"):
                    rng.append(f"before {match['versionEndExcluding']}")
                if match.get("versionEndIncluding"):
                    rng.append(f"through {match['versionEndIncluding']}")
                name = f"{vendor} {product}"
                if version not in {"*", "-"}:
                    name += f" {version}"
                if rng:
                    name += " (" + " ".join(rng) + ")"
                affected.append(name.strip())
    return list(dict.fromkeys(affected))[:10]


def _nvd_cve_document(item, fetched_at):
    cve_id = str(item.get("id") or "").strip().upper()
    descriptions = item.get("descriptions") or []
    description = next(
        (str(d.get("value")) for d in descriptions if d.get("lang") == "en" and d.get("value")),
        str(descriptions[0].get("value")) if descriptions else "No English description supplied."
    )
    metrics = item.get("metrics") or {}
    cvss_score, cvss_severity, cvss_version, cvss_vector = "", "", "", ""
    for key, version in (("cvssMetricV31", "3.1"), ("cvssMetricV30", "3.0"), ("cvssMetricV2", "2.0")):
        entries = metrics.get(key) or []
        if entries:
            data = entries[0].get("cvssData") or {}
            cvss_score = str(data.get("baseScore") or "")
            cvss_severity = str(data.get("baseSeverity") or entries[0].get("baseSeverity") or "")
            cvss_vector = str(data.get("vectorString") or "")
            cvss_version = version
            break
    weaknesses = []
    for weakness in item.get("weaknesses") or []:
        for desc in weakness.get("description") or []:
            if desc.get("lang") == "en" and desc.get("value"):
                weaknesses.append(str(desc["value"]))
    references = [
        str(ref.get("url"))
        for ref in (item.get("references") or [])
        if ref.get("url")
    ][:4]
    kev_name = item.get("cisaVulnerabilityName")
    kev_added = item.get("cisaExploitAdd")
    kev_due = item.get("cisaActionDue")
    kev_action = item.get("cisaRequiredAction")
    published = str(item.get("published") or "Not supplied")
    modified = str(item.get("lastModified") or "Not supplied")
    affected = _nvd_affected_products(item)

    fields = [
        f"CVE ID: {cve_id}",
        f"NVD description: {description}",
        f"Published: {published}",
        f"Last modified: {modified}",
    ]
    if affected:
        fields.append("Affected products (NVD configurations): " + "; ".join(affected[:8]))
    if cvss_score:
        fields.append(
            f"CVSS v{cvss_version}: {cvss_score} ({cvss_severity or 'severity not supplied'}) {cvss_vector}".strip()
        )
    if weaknesses:
        fields.append("Weakness (CWE): " + ", ".join(dict.fromkeys(weaknesses)))
    if kev_name or kev_added:
        fields.append(
            f"CISA KEV: listed as '{kev_name or 'KEV entry'}' (added {kev_added or 'date not supplied'}, "
            f"remediation due {kev_due or 'date not supplied'}). Required action: {kev_action or 'Not supplied'}"
        )
    else:
        fields.append("CISA KEV: not listed in the KEV section of this NVD record.")
    if references:
        fields.append("NVD references: " + " | ".join(references))
    fields.append(
        "Source channel: NIST National Vulnerability Database live API "
        "(authoritative CVE record fetched at request time, not a static dataset snapshot)."
    )

    return Document(
        page_content="\n".join(fields),
        metadata={
            "source_dataset": "nist/nvd",
            "source_kind": "nvd_api",
            "record_type": "live_cve_record",
            "title": f"{cve_id} — NVD record",
            "record_id": cve_id,
            "record_date": modified,
            "source_url": f"https://nvd.nist.gov/vuln/detail/{cve_id}",
            "fetched_at": fetched_at,
            "cvss_score": cvss_score,
            "cvss_severity": cvss_severity,
            "cvss_version": cvss_version,
            "in_kev": bool(kev_added),
            "kev_name": kev_name or "",
            "description": description,
            "published": published,
            "affected": affected,
            "weaknesses": weaknesses,
            "cvss_vector": cvss_vector,
            "references": references,
        }
    )


def fetch_nvd_cve_records(cve_ids, max_ids=3):
    """Fetch authoritative NVD records for CVE IDs missing from the local snapshot.

    Returns (docs, status_text). NVD is a free public API; without a key it allows
    roughly 5 requests per 30 seconds, so multi-CVE lookups are spaced out.
    """
    requested = [str(c).strip().upper() for c in dict.fromkeys(cve_ids or []) if str(c).strip()][:max_ids]
    docs_out, statuses = [], []
    for position, cve_id in enumerate(requested):
        cached = _NVD_CACHE.get(cve_id)
        if cached and (time.time() - cached["cached_at"]) < _NVD_CACHE_TTL_SECONDS:
            docs_out.append(cached["doc"])
            statuses.append(f"{cve_id}: served from the 30-minute NVD cache")
            continue
        if position > 0:
            time.sleep(6)  # stay inside the unauthenticated NVD rate limit
        try:
            response = requests.get(
                _NVD_API_URL,
                params={"cveId": cve_id},
                headers={"Accept": "application/json", "User-Agent": "CTI-RAG-Student-Project/1.0"},
                timeout=(5, 15),
            )
            fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
            if response.status_code == 404:
                statuses.append(f"{cve_id}: not present in NVD")
                continue
            response.raise_for_status()
            vulns = (response.json() or {}).get("vulnerabilities") or []
            if not vulns:
                statuses.append(f"{cve_id}: no record returned by NVD")
                continue
            doc = _nvd_cve_document(vulns[0].get("cve") or {}, fetched_at)
            _NVD_CACHE[cve_id] = {"cached_at": time.time(), "doc": doc}
            docs_out.append(doc)
            statuses.append(f"{cve_id}: fetched live from NVD at {fetched_at} UTC")
        except (requests.RequestException, ValueError) as exc:
            statuses.append(f"{cve_id}: NVD lookup failed ({type(exc).__name__}: {str(exc)[:160]})")
    status = ("NVD live fallback: " + "; ".join(statuses)) if statuses else ""
    return docs_out, status


print("ThreatCluster live API connector ready. Add THREATCLUSTER_API_KEY to enable it.")


ThreatCluster live API connector ready. Add THREATCLUSTER_API_KEY to enable it.


# Evaluation & Testing

Run the evaluation cell below **before** launching the Gradio interface. It records retrieval latency, Gemma generation latency and total response time for historical, vulnerability and ransomware-target queries.

After reviewing each answer against its retrieved evidence, fill in the manual scores:
- **Correctness:** 0 = incorrect, 1 = partly correct, 2 = supported/correct.
- **Grounding:** 0 = unsupported, 1 = partly supported, 2 = all material claims supported by the records.
- **Relevance:** 0 = does not answer the question, 1 = partly relevant, 2 = directly relevant.

The CVE and ransomware sources are snapshots, not live databases. Do not invent or estimate evaluation scores, financial losses or breach confirmation.

Strict grounding check: verify that every case-specific statement, impact, financial-loss figure, and mitigation is present in the selected evidence. A missing field must be marked "Not found in the selected datasets or live API results." Do not award correctness for plausible facts supplied from model memory.

## Email-triage behaviour and safety boundary

The email workflow combines (1) transparent text indicators, (2) exact domain matching against the separately downloaded PhishDestroy feed, and (3) RAG retrieval for historical context. Domain matches are leads, not proof; non-matches do not mean safe. The feed is checked once when this notebook starts, so restart/re-run the feed-loading cell to refresh it.

The answer separates the likely attacker objective, identifiable target, potential impact and actual loss. Actual loss is left as unknown unless a source documents it. No links, attachments or scripts are opened or executed.

In [8]:
import time
import pandas as pd

# These questions test historical, current-snapshot and structured-record retrieval.
latest_cve_id = ""
if not cve_df.empty and "cve_id" in cve_df.columns:
    latest_cve_rows = cve_df.sort_values("published_date", ascending=False) if "published_date" in cve_df.columns else cve_df
    latest_cve_id = str(latest_cve_rows.iloc[0]["cve_id"])

latest_cve_question = (
    f"For {latest_cve_id}, what does the loaded snapshot say about severity and exploitation?"
    if latest_cve_id else
    "What vulnerability fields are available in the loaded CVE snapshot?"
)

evaluation_questions = [
    "What do historical reports state about MiniDuke PDF zero-day attacks and its micro backdoor?",
    latest_cve_question,
    "What ransomware-group victim claims are recorded for healthcare organisations, and what does the dataset actually establish?"
]

evaluation_rows = []
for question in evaluation_questions:
    retrieval_start = time.perf_counter()
    retrieved = hybrid_retrieve(question)
    structured = retrieve_structured_context(question)
    case_docs = retrieve_incident_case_studies(question, max_cases=2)
    case_context = format_docs(case_docs, max_chars_per_doc=1400) if case_docs else "No matching named incident case was found in the loaded incident summaries."
    retrieval_seconds = time.perf_counter() - retrieval_start

    rag_start = time.perf_counter()
    rag_answer = qa_chain.invoke({
        "context": format_docs(retrieved),
        "structured_context": structured,
        "case_study_context": case_context,
        "input": question,
        "response_instruction": "QUESTION TYPE: Direct CTI Q&A. Answer the exact question in a concise paragraph or bullets. Cite every factual claim and mark missing facts as 'Not found in the selected datasets or live API results.'"
    })
    generation_seconds = time.perf_counter() - rag_start

    evaluation_rows.append({
        "question": question,
        "retrieved_chunks": len(retrieved),
        "incident_cases_selected": len(case_docs),
        "retrieval_seconds": round(retrieval_seconds, 3),
        "generation_seconds": round(generation_seconds, 3),
        "total_seconds": round(retrieval_seconds + generation_seconds, 3),
        "answer": rag_answer,
        "human_correctness_0_to_2": None,
        "grounding_0_to_2": None,
        "review_notes": "Manually verify each claim against the cited source records."
    })

evaluation_results = pd.DataFrame(evaluation_rows)
display(evaluation_results)
evaluation_results.to_csv("cti_multisource_evaluation_results.csv", index=False)
print("Saved cti_multisource_evaluation_results.csv")

,question,retrieved_chunks,incident_cases_selected,retrieval_seconds,generation_seconds,total_seconds,answer,human_correctness_0_to_2,grounding_0_to_2,review_notes
0,What do historical reports state about MiniDuk...,8,2,2.674,43.993,46.666,Historical reports state that MiniDuke PDF zer...,None,None,Manually verify each claim against the cited s...
1,"For CVE-2026-86151, what does the loaded snaps...",8,2,2.305,5.055,7.359,The CVE-2026-86151 vulnerability is a critical...,None,None,Manually verify each claim against the cited s...
2,What ransomware-group victim claims are record...,8,2,1.526,10.946,12.472,The dataset records victim claims for ransomwa...,None,None,Manually verify each claim against the cited s...


Saved cti_multisource_evaluation_results.csv


## Response-speed and progress improvements

- The first query is warmed up when Gemma loads, so one-time CUDA initialisation is less likely to affect the first analyst request.
- Gemma uses KV caching, greedy decoding, one beam and a 210-token response cap. Increase `MAX_NEW_TOKENS` to 140 if answers are cut off; this may increase latency.
- Email triage retrieves with a short intent query and skips CVE/ransomware table scans that rarely help classify a pasted email.
- Structured table matching uses one vectorised regex pass instead of a separate full-table scan for every term.
- Gradio shows a progress indicator and status descriptions while the request runs. Requests are serialised to avoid GPU contention.
- Measure retrieval and generation separately. The actual latency depends on the selected Colab GPU and current runtime load.


## Named incident case studies

For broad sector/attack questions (for example healthcare ransomware), the app now selects up to two distinct incident summaries from the incident-cluster dataset. It separates case-specific facts, documented response, actual losses and recommended mitigations. Ransomware leak-site victim rows are not treated as confirmed incidents. If the incident snapshot has no matching named cases, the app must say so rather than inventing examples.


## Approved-source factuality / abstention
- CTI research answers may only use supplied records from the selected datasets and, when enabled, the separately labelled live ThreatCluster API.
- The notebook includes a pre-generation relevance gate. When there is no sufficiently matching evidence, it returns a fixed abstention response without calling Gemma.
- For broad sector + attack questions, named cases must come from the selected incident-summary dataset. If no matching named case is found, the assistant says so and does not import famous cases from general knowledge.
- Within an otherwise relevant case, unavailable target, method, impact, loss, response, and mitigation fields are each reported as "Not found in the selected datasets or live API results."
- The prompt forbids generic mitigation advice from model memory. Mitigation is reported only when a selected dataset explicitly contains it.
- For email triage, the submitted email is treated as direct input evidence for visible warning signs. Campaign attribution, threat-actor names, confirmed incidents, losses, and mitigation must still be supported by selected dataset records or the downloaded domain feed.


## Grounded-answer quality and latency safeguards (revision)

- Exact CVE identifiers are resolved through a precomputed, whitespace-normalised structured lookup index.
- Exact CVE metadata questions are answered deterministically from the selected CVE snapshot with a [V#] citation; Gemma and the live API are bypassed for this path.
- The legacy frontend's forced four-section formatting directive is stripped before retrieval; the website should also send the analyst's original question without appending that directive.
- Simple questions receive a short direct answer. Case studies, mitigations and step-by-step sections appear only when requested.
- Current/recent/live questions may use the optional ThreatCluster API. Static definitions and historical questions skip that network call.
- If an exact CVE record is missing or duplicate records conflict, the assistant abstains rather than choosing a loosely related semantic result.
- Generated CTI answers are withheld if they lack valid inline citations for every substantive sentence.


## Semantic-first query handling (updated)

This version does not use a fixed list of trigger words to decide whether a CTI question deserves incident retrieval or live enrichment.

- Natural-language questions are searched with the existing Chroma semantic index plus BM25 exact-term retrieval.
- Incident/case-study intent is classified using sentence embeddings against intent descriptions; incident records are retrieved by semantic similarity from the existing Chroma index.
- Hand-written keyword expansion has been removed. The original analyst question is preserved for retrieval.
- Live ThreatCluster enrichment is controlled by the user's opt-in checkbox, not by detecting words such as “latest” or “current”. The API currently searches a seven-day window, so it supplements rather than replaces historical snapshots.
- Citation validation gets one constrained repair attempt. If valid citations still cannot be produced, the assistant explains that relevant records may exist and exposes them in the Sources panel instead of falsely claiming that no data exists.
- The live API checkbox is off by default to reduce latency and avoid unnecessary API-credit use.


In [9]:
import gradio as gr
import time
import re

def _source_record_markdown(doc, label):
    meta = doc.metadata or {}
    if meta.get("source_kind") == "nvd_api":
        lines = [
            f"### [{label}] {meta.get('title', 'NVD CVE record')}",
            "**Source channel:** NIST National Vulnerability Database live API (authoritative CVE record, not a static dataset)",
            f"**NVD page:** [{meta.get('source_url', 'https://nvd.nist.gov/')}]({meta.get('source_url', 'https://nvd.nist.gov/')})",
            f"**CVE ID:** {meta.get('record_id', 'Not supplied')}",
            f"**Last modified by NVD:** {meta.get('record_date', 'Not supplied')}",
            f"**Fetched at (UTC):** {meta.get('fetched_at', 'Not supplied')}",
            "**Retrieved API evidence:**\n\n" + (doc.page_content or ""),
        ]
        return "\n\n".join(lines)
    if meta.get("source_kind") == "live_api":
        lines = [
            f"### [{label}] {meta.get('title', 'Live ThreatCluster record')}",
            "**Source channel:** ThreatCluster live REST API (not a static Hugging Face dataset)",
            f"**API endpoint:** `{meta.get('api_endpoint', '/search')}`",
            f"**Record ID:** {meta.get('record_id', 'Not supplied')}",
            f"**Record date returned by API:** {meta.get('record_date', 'Not supplied')}",
            f"**Fetched at (UTC):** {meta.get('fetched_at', 'Not supplied')}",
            f"**Request cost:** {meta.get('request_cost', 'Not returned')} credit(s)",
            f"**Credits remaining:** {meta.get('credits_remaining', 'Not returned')}",
            f"**API reference:** [{meta.get('api_docs_url', THREATCLUSTER_API_REFERENCE)}]({meta.get('api_docs_url', THREATCLUSTER_API_REFERENCE)})",
        ]
        if meta.get('source_url') and meta.get('source_url') != THREATCLUSTER_API_REFERENCE:
            lines.append(f"**Original report/source:** [{meta['source_url']}]({meta['source_url']})")
        lines.append("**Retrieved API evidence:**\n\n" + (doc.page_content or ""))
        return "\n\n".join(lines)
    dataset_name = meta.get("source_dataset", "Unknown dataset")
    info = _dataset_info(dataset_name)
    title = meta.get("title", "Untitled record")
    date = meta.get("record_date", "Date not supplied")
    record_id = meta.get("record_id", "Not supplied")
    original_url = meta.get("source_url", "")
    if original_url == info.get("url"):
        original_url = ""
    lines = [
        f"### [{label}] {title}",
        f"**Dataset:** `{dataset_name}`",
        f"**Dataset citation:** {info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Hugging Face.",
        f"**Dataset card:** [{info['url']}]({info['url']})" if info.get("url") else "**Dataset card:** Not supplied",
        f"**Record date:** {date}",
        f"**Record ID:** {record_id}",
        f"**Dataset note:** {info.get('note', '')}",
        f"**License shown on dataset card:** {info.get('license', 'Check dataset card')}"
    ]
    if original_url:
        lines.append(f"**Original report/article:** [{original_url}]({original_url})")
    lines.append("**Retrieved excerpt:**\n\n" + (doc.page_content or ""))
    return "\n\n".join(lines)


def build_sources_panel(retrieved_docs, structured_context, case_docs=None, include_phish_feed=False, live_docs=None, live_status="", nvd_docs=None, nvd_status=""):
    sections = []
    if live_status:
        sections.append("### Live API status\n\n" + live_status)
    if nvd_status:
        sections.append("### NVD live lookup status\n\n" + nvd_status)
    for index, doc in enumerate(live_docs or [], start=1):
        sections.append(_source_record_markdown(doc, f"L{index}"))
    for index, doc in enumerate(nvd_docs or [], start=1):
        sections.append(_source_record_markdown(doc, f"N{index}"))
    for index, doc in enumerate(case_docs or [], start=1):
        sections.append(_source_record_markdown(doc, f"C{index}"))
    for index, doc in enumerate(retrieved_docs or [], start=1):
        sections.append(_source_record_markdown(doc, f"S{index}"))

    if structured_context and structured_context.strip() not in {
        _NO_STRUCTURED, "Not applicable to this email-only check."
    }:
        sections.append("### Structured selected-dataset records\n\n" + structured_context)

    if include_phish_feed:
        info = _dataset_info("phishdestroy/destroylist")
        sections.append(
            "### [IOC1] PhishDestroy domain-feed source\n"
            "**Dataset citation:** phishdestroy. (n.d.). *destroylist* [Data set]. Hugging Face.\n\n"
            f"**Dataset card:** [{info['url']}]({info['url']})\n\n"
            f"**Feed timestamp reported by publisher:** {IOC_FEED_LAST_UPDATED}\n\n"
            f"**Feed status:** {IOC_FEED_STATUS}"
        )
    if not sections:
        return "No supporting records were retrieved from the selected datasets."
    return "\n\n---\n\n".join(sections)


def build_answer_citations(
    retrieved_docs, case_docs=None, structured_context="", include_phish_feed=False,
    live_docs=None, answer_text="", nvd_docs=None
):
    used = set(re.findall(r"\[((?:S|C|V|R|L|N|AGG)\d+|IOC1)\]", answer_text or ""))
    lines = []

    for label in dict.fromkeys(re.findall(r"\[(AGG\d+)\]", answer_text or "")):
        if label not in used:
            continue
        lines.append(
            f"- **[{label}] Derived dataset statistic** — aggregate counted at request time from the "
            "dataset named in the Structured selected-dataset records block; not a single stored record."
        )

    for index, doc in enumerate(nvd_docs or [], start=1):
        label = f"N{index}"
        if label not in used:
            continue
        meta = doc.metadata or {}
        nvd_url = meta.get("source_url", "https://nvd.nist.gov/")
        lines.append(
            f"- **[{label}] {meta.get('title', 'NVD CVE record')}** — NIST National Vulnerability Database "
            f"live API record; fetched {meta.get('fetched_at', 'time not supplied')} UTC; NVD page: {nvd_url}"
        )

    for index, doc in enumerate(live_docs or [], start=1):
        label = f"L{index}"
        if label not in used:
            continue
        meta = doc.metadata or {}
        source_url = meta.get("source_url", THREATCLUSTER_API_REFERENCE)
        item = f"- **[{label}] {meta.get('title', 'Live ThreatCluster record')}** — ThreatCluster live API result; fetched {meta.get('fetched_at', 'time not supplied')} UTC; API reference: {meta.get('api_docs_url', THREATCLUSTER_API_REFERENCE)}"
        if source_url and source_url != THREATCLUSTER_API_REFERENCE:
            item += f"; original report/source: {source_url}"
        lines.append(item)

    for index, doc in enumerate(case_docs or [], start=1):
        label = f"C{index}"
        if label not in used:
            continue
        meta = doc.metadata or {}
        info = _dataset_info(meta.get("source_dataset", "Unknown dataset"))
        original = meta.get("source_url", "")
        item = f"- **[{label}] {meta.get('title', 'Untitled record')}** — {info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Dataset card: {info['url']}"
        if original and original != info.get("url"):
            item += f"; original report/source: {original}"
        lines.append(item)

    for index, doc in enumerate(retrieved_docs or [], start=1):
        label = f"S{index}"
        if label not in used:
            continue
        meta = doc.metadata or {}
        info = _dataset_info(meta.get("source_dataset", "Unknown dataset"))
        original = meta.get("source_url", "")
        item = f"- **[{label}] {meta.get('title', 'Untitled record')}** — {info['owner']}. ({info['year']}). *{info['title']}* [Data set]. Dataset card: {info['url']}"
        if original and original != info.get("url"):
            item += f"; original report/source: {original}"
        lines.append(item)

    for label in dict.fromkeys(re.findall(r"\[(V\d+)\]", structured_context or "")):
        if label not in used:
            continue
        info = _dataset_info("threatcluster/cve-exploitation-signals")
        lines.append(
            f"- **[{label}] CVE structured record** — {info['owner']}. ({info['year']}). "
            f"*{info['title']}* [Data set]. Dataset card: {info['url']}"
        )

    for label in dict.fromkeys(re.findall(r"\[(R\d+)\]", structured_context or "")):
        if label not in used:
            continue
        info = _dataset_info("threatcluster/ransomware-leak-site-victims")
        lines.append(
            f"- **[{label}] Ransomware listing record** — {info['owner']}. ({info['year']}). "
            f"*{info['title']}* [Data set]. Dataset card: {info['url']}"
        )

    if include_phish_feed and "IOC1" in used:
        info = _dataset_info("phishdestroy/destroylist")
        lines.append(
            f"- **[IOC1] Domain-feed check** — {info['owner']}. ({info['year']}). "
            f"*{info['title']}* [Data set]. Dataset card: {info['url']}"
        )

    return "\n\n### Sources cited\n" + "\n".join(lines) if lines else ""


_NOT_FOUND = "Not found in the selected datasets or live API results."
_NO_STRUCTURED = "No matching structured CVE or ransomware-target records were selected for this query."


def resolve_followup_query(user_query, previous_context=None):
    """Resolve short anaphoric follow-ups like 'mitigation for that?' against prior turn context."""
    q = str(user_query or "").strip()
    if not q or not previous_context:
        return q
    prev_q = str(previous_context.get("last_query") or "").strip()
    prev_a = str(previous_context.get("last_answer") or "").strip()
    if not prev_q:
        return q

    has_anaphora = bool(re.search(
        r"\b(that|it|this|these|those|them|their|its|same|above|vulnerability|attack|incident|malware)\b",
        q.lower()
    ))
    is_short_followup = len(q.split()) <= 10 and (
        has_anaphora or re.match(r"^(and\s+|what about|how about|miti?gat\w*|mitgat\w*|remediat\w*|fix|impact|response)", q.lower())
    )
    if not is_short_followup:
        return q

    # Extract key entities (CVEs or subject nouns) from previous turn
    cves = list(dict.fromkeys(m.upper() for m in re.findall(r"\bCVE-\d{4}-\d{3,}\b", prev_q + " " + prev_a, re.I)))
    if cves:
        return f"{q} (Follow-up regarding {', '.join(cves)} from previous question: {prev_q[:220]})"
    return f"{q} (Follow-up to previous question: {prev_q[:240]})"


def query_response_style(query):
    low = str(query or "").lower()
    wants_case = semantic_case_request(query)
    wants_mit = bool(re.search(r"\b(miti?gat\w*|mitgat\w*|remediat\w*|defen[cs]\w*|prevent\w*|protect\w*|countermeasure\w*|workaround\w*|harden\w*|patch\w*)\b", low))
    wants_def = bool(re.search(r"\b(what is|define|definition|explain|how it works|how does)\b", low))

    if wants_case and (wants_def or wants_mit):
        return "multi_part_case"
    if wants_case:
        return "case_studies"
    if wants_def and wants_mit:
        return "definition_mitigation"
    if wants_mit:
        return "mitigation"
    if wants_def:
        return "definition"
    return "direct_answer"


def response_instruction_for(query):
    style = query_response_style(query)
    if style == "multi_part_case":
        return (
            "QUESTION TYPE: Multi-part explanation + case study + mitigations. "
            "Use short bold section headers for each requested part (e.g., **Overview / How It Works**, **Documented Case Example**, **Documented Mitigations**). "
            "Use [S#] records for technical definitions, mechanics, and documented mitigations, and [C#], [S#], [N#], or [L#] records for real-world case examples. "
            "If any requested part (such as a similar case or step-by-step mitigation) is absent from the supplied records, write 'Not found in the selected datasets or live API results.' under that part. "
            "End EVERY factual sentence or bullet item with its exact evidence label."
        )
    if style == "definition_mitigation":
        return (
            "QUESTION TYPE: Definition plus mitigation. Do NOT show documented cases unless asked. "
            "Use two short sections: **Definition** (1-3 sentences from records) and **Documented Mitigations** (bullet only actions explicitly present in records; otherwise write 'Not found in the selected datasets or live API results.'). "
            "End EVERY factual sentence or bullet item with its exact [S#], [V#], [N#], or [L#] label."
        )
    if style == "definition":
        return "QUESTION TYPE: Definition. Give a direct, concise 1-3 sentence definition supported by records, ending every factual sentence with its exact [S#], [V#], [N#], or [L#] label."
    if style == "mitigation":
        return "QUESTION TYPE: Mitigation. List only mitigations explicitly recorded in supplied data [S#], [V#], [N#], or [L#]; otherwise say 'Not found in the selected datasets or live API results.' End every bullet or sentence with its exact evidence label."
    if style == "case_studies":
        return (
            "QUESTION TYPE: Case study / real-world example. Prefer named incident-cluster records [C#] or live records [L#], or [S#] historical/curated records if they directly describe the requested case. "
            "Separate each requested case clearly with bold labels, covering what happened, impact, and documented response only as stated in the records. "
            "For missing impact, financial loss, or response details say 'Not found in the selected datasets or live API results.' "
            "End EVERY factual sentence or bullet with its exact [C#], [S#], [N#], or [L#] label."
        )
    return "QUESTION TYPE: Direct CTI Q&A. Answer the exact question concisely, ending every factual sentence or bullet with its exact citation label, and mark missing facts as 'Not found in the selected datasets or live API results.'"


def has_relevant_selected_evidence(query, retrieved_docs, structured_context, case_docs=None, live_docs=None, nvd_docs=None):
    q = str(query or "").strip()
    if not q:
        return False, []

    docs_all = list(case_docs or []) + list(retrieved_docs or []) + list(live_docs or []) + list(nvd_docs or [])
    parts = [(doc.page_content or "").lower() for doc in docs_all]
    if structured_context and structured_context.strip() not in {
        _NO_STRUCTURED, "Not applicable to this email-only check."
    }:
        parts.append(str(structured_context).lower())
    evidence = "\n".join(parts)
    if not evidence.strip():
        return False, []

    requested_cves = [c.upper() for c in re.findall(r"\bCVE-\d{4}-\d{3,}\b", q, re.I)]
    if requested_cves:
        missing = [c for c in requested_cves if c.lower() not in evidence]
        if missing:
            return False, []
        return True, requested_cves

    named_terms = []
    for phrase in ("miniduke", "mini duke", "apt29", "cozy bear", "solarwinds",
                   "wannacry", "notpetya", "not petya", "stuxnet", "conficker",
                   "emotet", "trickbot", "lockbit", "conti", "ryuk"):
        if re.search(rf"(?<![a-z0-9]){re.escape(phrase)}(?![a-z0-9])", q.lower()):
            named_terms.append(phrase)
    if named_terms and not any(term in evidence for term in named_terms):
        return False, []

    concept_groups = {
        "command_and_control": ["command-and-control", "command and control", "c2", "c&c"],
        "obfuscation": ["obfuscation", "obfuscated", "packing", "packed", "encoding", "encoded"],
        "encryption": ["encryption", "encrypted", "cipher", "tls", "ssl", "https", "aes", "rc4", "rsa"],
        "zero_day": ["zero-day", "zero day", "0-day", "0 day", "unpatched", "cve-2013-0640"],
        "pdf_exploit": ["pdf", "adobe reader", "acrobat", "malicious document"],
        "ransomware": ["ransomware", "file encryption", "double extortion", "data extortion"],
        "phishing": ["phishing", "spear phishing", "credential theft", "social engineering"],
        "wiper": ["wiper", "disk wiper", "data destruction", "destructive malware", "mbr", "shamoon", "hermeticwiper", "whispergate"],
    }

    matched_groups = []
    q_low = q.lower()
    for group, aliases in concept_groups.items():
        if any(re.search(rf"(?<![a-z0-9]){re.escape(a)}(?![a-z0-9])", q_low) for a in aliases):
            if any(re.search(rf"(?<![a-z0-9]){re.escape(a)}(?![a-z0-9])", evidence) for a in aliases):
                matched_groups.append(group)

    generic = set(_STOP_WORDS) | {
        "case", "cases", "study", "studies", "example", "examples", "similar",
        "actual", "reported", "reporting", "known", "using", "dataset", "selected",
        "organisation", "organisations", "organization", "organizations",
        "please", "could", "would", "should", "state", "states", "regarding",
        "information", "compare", "comparison", "techniques", "technique",
        "protocol", "protocols", "methods", "method", "explain", "explains",
        "explanation", "works", "work", "step", "steps", "detail", "details",
        "list", "give", "describe", "what", "how", "why", "which", "the",
        "follow-up", "previous", "question", "mitigation", "mitigations"
    }
    tokens = [
        t for t in re.findall(r"[a-z0-9][a-z0-9._-]{1,}", q_low)
        if t not in generic and not t.isdigit()
    ]
    matched_tokens = [
        t for t in dict.fromkeys(tokens)
        if re.search(rf"(?<![a-z0-9]){re.escape(t)}(?![a-z0-9])", evidence)
    ]

    if matched_groups:
        return True, matched_groups + matched_tokens[:8]

    if tokens:
        required = 1 if len(tokens) <= 4 else 2
        if len(matched_tokens) >= required:
            return True, matched_tokens

    try:
        query_vector = np.asarray(embeddings.embed_query(q), dtype=np.float32)
        query_vector /= max(float(np.linalg.norm(query_vector)), 1e-12)
        candidate_texts = [
            (doc.page_content or "")[:2500]
            for doc in docs_all if (doc.page_content or "").strip()
        ]
        if candidate_texts:
            candidate_vectors = np.asarray(
                embeddings.embed_documents(candidate_texts), dtype=np.float32
            )
            candidate_vectors /= np.maximum(
                np.linalg.norm(candidate_vectors, axis=1, keepdims=True), 1e-12
            )
            best_score = float(np.max(candidate_vectors @ query_vector))
            if best_score >= 0.30:
                return True, [f"semantic_match:{best_score:.3f}"]
    except Exception:
        pass

    return False, matched_tokens


def dataset_only_no_answer(reason, retrieval_seconds=0.0, total_seconds=0.0):
    return (
        f"**{_NOT_FOUND}**\n\n"
        f"{reason}\n\n"
        f"*Processing: retrieval {retrieval_seconds:.2f}s · total {total_seconds:.2f}s.*"
    )


def _clean_injected_frontend_instructions(question):
    text = str(question or "").strip()
    text = re.sub(r"\s*\[Formatting Requirement:.*$", "", text, flags=re.I | re.S)
    return text.strip()


def _truthy_dataset_value(value):
    if value is None:
        return None
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y", "listed", "in kev"}:
        return True
    if text in {"false", "0", "no", "n", "not listed", "not in kev"}:
        return False
    return None


def build_direct_cve_answer(query, records, requested_ids, retrieval_seconds, total_start, with_footer=True):
    if not records:
        elapsed = time.perf_counter() - total_start
        return dataset_only_no_answer(
            "No exact matching CVE record was found in the selected CVE exploitation-signals dataset.",
            retrieval_seconds, elapsed
        )

    lines = []
    low = query.lower()
    for index, cve_id in enumerate(requested_ids, start=1):
        matches = [
            row for row in records
            if str(row.get("_cve_id_norm") or row.get("cve_id") or "").strip().upper() == cve_id
        ]
        if not matches:
            elapsed = time.perf_counter() - total_start
            return dataset_only_no_answer(
                f"No exact matching record for {cve_id} was found in the selected CVE exploitation-signals dataset.",
                retrieval_seconds, elapsed
            )

        descriptions = list(dict.fromkeys(
            as_text(row.get("description"), 900).strip()
            for row in matches if as_text(row.get("description"), 900).strip()
        ))
        label = f"[V{index}]"
        if len(descriptions) > 1:
            elapsed = time.perf_counter() - total_start
            return (
                f"**The selected dataset contains conflicting descriptions for {cve_id}; "
                f"I cannot determine the correct description from these records.** {label}\n\n"
                f"*Processing: retrieval {retrieval_seconds:.2f}s · total {elapsed:.2f}s.*"
            )
        row = matches[0]
        description = descriptions[0] if descriptions else ""
        if not description:
            elapsed = time.perf_counter() - total_start
            return dataset_only_no_answer(
                f"The selected record for {cve_id} does not contain a description.",
                retrieval_seconds, elapsed
            )

        lines.append(f"**{cve_id}:** {description} {label}")

        if re.search(r"\b(cvss|severity|score)\b", low):
            score = as_text(row.get("cvss_v3_score"), 40).strip()
            severity = as_text(row.get("cvss_v3_severity"), 40).strip()
            value = " / ".join(x for x in [score, severity] if x and x.lower() not in {"nan", "none"})
            lines.append(f"**CVSS v3:** {value} {label}" if value else f"**CVSS v3:** {_NOT_FOUND}")

        if re.search(r"\b(kev|known exploited|catalogue|catalog)\b", low):
            kev = _truthy_dataset_value(row.get("in_kev"))
            value = "Yes" if kev is True else "No" if kev is False else _NOT_FOUND
            lines.append(f"**CISA KEV in this dataset snapshot:** {value} {label if kev is not None else ''}".rstrip())

        if re.search(r"\b(affected product|product|vendor|manufacturer)\b", low):
            vendor = as_text(row.get("affected_vendors"), 500).strip()
            product = as_text(row.get("affected_products"), 500).strip()
            value = "; ".join(x for x in [vendor, product] if x and x.lower() not in {"nan", "none"})
            lines.append(f"**Affected vendor/product:** {value} {label}" if value else f"**Affected vendor/product:** {_NOT_FOUND}")

        if re.search(r"\b(exploit|exploited|exploitation)\b", low):
            has_exploit = _truthy_dataset_value(row.get("has_exploit"))
            count = as_text(row.get("exploit_count"), 40).strip()
            if has_exploit is None:
                lines.append(f"**Public exploit recorded in this snapshot:** {_NOT_FOUND}")
            else:
                detail = "Yes" if has_exploit else "No"
                if count and count.lower() not in {"nan", "none"}:
                    detail += f" (recorded exploit count: {count})"
                lines.append(f"**Public exploit recorded in this snapshot:** {detail} {label}")

        if re.search(r"\b(ransomware use|used by ransomware|ransomware)\b", low):
            ransomware_use = _truthy_dataset_value(row.get("ransomware_use"))
            value = "Yes" if ransomware_use is True else "No" if ransomware_use is False else _NOT_FOUND
            lines.append(f"**Ransomware use in this dataset snapshot:** {value} {label if ransomware_use is not None else ''}".rstrip())

        if re.search(r"\b(published|publication date|published date)\b", low):
            date_value = as_text(row.get("published_date"), 50).strip()
            valid_date = date_value and date_value.lower() not in {"nan", "none"}
            lines.append(f"**Published date:** {date_value} {label}" if valid_date else f"**Published date:** {_NOT_FOUND}")

    elapsed = time.perf_counter() - total_start
    body = "\n\n".join(lines)
    if with_footer:
        body += f"\n\n*Processing: structured CVE lookup {retrieval_seconds:.2f}s · total {elapsed:.2f}s (Gemma not required).*"
    return body


def _cvss_vector_summary(vector):
    """Decode a CVSS vector string into a plain-English exploitation profile."""
    if not vector or "/" not in str(vector):
        return ""
    metrics = {}
    for part in str(vector).split("/"):
        if ":" in part:
            k, v = part.split(":", 1)
            metrics[k.strip().upper()] = v.strip().upper()
    bits = []
    if metrics.get("AV") == "N":
        bits.append("exploitable remotely over the network")
    elif metrics.get("AV") == "A":
        bits.append("exploitable from an adjacent network segment")
    elif metrics.get("AV") == "L":
        bits.append("requires local access to exploit")
    elif metrics.get("AV") == "P":
        bits.append("requires physical access to exploit")
    if metrics.get("AC") == "L":
        bits.append("low attack complexity")
    if metrics.get("PR") == "N":
        bits.append("no privileges or authentication needed")
    if metrics.get("UI") == "N":
        bits.append("no user interaction needed")
    if metrics.get("S") == "C":
        bits.append("impact can extend beyond the vulnerable component")
    impacts = [name for name, key in (("confidentiality", "C"), ("integrity", "I"), ("availability", "A")) if metrics.get(key) == "H"]
    if impacts:
        bits.append("high impact on " + ", ".join(impacts))
    return ("; ".join(bits).capitalize() + ".") if bits else ""


def build_direct_nvd_answer(query, nvd_docs):
    """Deterministic answer built from a live NVD record for CVEs missing from the snapshot."""
    lines = []
    low = query.lower()
    for index, doc in enumerate(nvd_docs or [], start=1):
        meta = doc.metadata or {}
        label = f"[N{index}]"
        cve_id = meta.get("record_id", "CVE record")
        description = str(meta.get("description") or "").strip()
        if description:
            lines.append(f"**{cve_id} (live NVD record):** {description} {label}")
        else:
            lines.append(f"**{cve_id} (live NVD record):** {_NOT_FOUND} {label}")

        affected = [a for a in (meta.get("affected") or []) if a]
        if affected:
            lines.append(f"**Affected products/versions (NVD):** " + "; ".join(affected[:6]) + f" {label}")

        summary = _cvss_vector_summary(meta.get("cvss_vector"))
        if summary:
            lines.append(f"**How it can be exploited:** {summary} {label}")

        weaknesses = [w for w in (meta.get("weaknesses") or []) if w]
        if weaknesses:
            lines.append(f"**Weakness type (CWE):** " + ", ".join(weaknesses[:4]) + f" {label}")

        score = str(meta.get("cvss_score") or "").strip()
        severity = str(meta.get("cvss_severity") or "").strip()
        version = str(meta.get("cvss_version") or "").strip()
        if score:
            lines.append(f"**CVSS v{version}:** {score} ({severity or 'severity not supplied'}) {label}")
        elif re.search(r"\b(cvss|severity|score)\b", low):
            lines.append(f"**CVSS:** {_NOT_FOUND}")

        published = str(meta.get("published") or "").strip()
        lines.append(f"**Published (NVD):** {published or _NOT_FOUND} {label}")

        if meta.get("in_kev"):
            lines.append(f"**CISA KEV:** Yes — listed as '{meta.get('kev_name') or 'KEV entry'}' in this NVD record {label}")
        else:
            lines.append(f"**CISA KEV:** Not listed in the KEV section of this NVD record {label}")

        references = [r for r in (meta.get("references") or []) if r]
        if references:
            lines.append(f"**NVD references:** " + " | ".join(references[:3]) + f" {label}")

    return "\n\n".join(lines)


def _normalize_citations(answer, available):
    """Expand grouped citations like [S1, S2] -> [S1] [S2] and drop non-existent labels."""
    def _expand(match):
        tokens = re.findall(r"(?:S|C|V|R|L|N)\d+", match.group(0))
        valid = [f"[{t}]" for t in tokens if t in available]
        return " ".join(valid)
    return re.sub(r"\[(?:S|C|V|R|L|N)\d+(?:\s*,\s*(?:S|C|V|R|L|N)\d+)*\]", _expand, answer or "")


def _answer_has_complete_valid_citations(answer, retrieved_docs, case_docs=None, structured_context="", live_docs=None, nvd_docs=None):
    """Validate that substantive claims carry valid citations without rejecting numbered lists, headers, or abstentions."""
    available = set()
    available.update(f"S{i}" for i, _ in enumerate(retrieved_docs or [], start=1))
    available.update(f"C{i}" for i, _ in enumerate(case_docs or [], start=1))
    available.update(f"L{i}" for i, _ in enumerate(live_docs or [], start=1))
    available.update(f"N{i}" for i, _ in enumerate(nvd_docs or [], start=1))
    available.update(re.findall(r"\[(V\d+|R\d+)\]", structured_context or ""))

    cleaned = _normalize_citations(answer, available)
    used = set(re.findall(r"\[((?:S|C|V|R|L|N)\d+)\]", cleaned))
    if not used or not used.issubset(available):
        return False

    abstention_phrases = (
        _NOT_FOUND.lower(),
        "not found in the selected datasets",
        "not reported in the retrieved",
        "no matching",
        "absent from the supplied",
        "not explicitly documented",
        "not stated in the retrieved",
    )

    for raw_line in cleaned.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or line in {"---", "***"}:
            continue
        if line.lower().startswith("*processing:"):
            continue
        # Skip standalone bold headers (e.g. **Defensive Mitigations:**) or list introducers ending in ':'
        if re.match(r"^\*\*[^*]+\*\*:?$", line) or line.endswith(":"):
            continue
        if any(phrase in line.lower() for phrase in abstention_phrases):
            continue

        # Strip leading list markers ('1. ', '- ', '* ') BEFORE splitting sentences so '1.' doesn't break validation
        body_line = re.sub(r"^(?:\d+[.)]|[-*•])\s+", "", line).strip()
        if not body_line:
            continue

        # If the whole bullet/line carries a valid citation, accept it
        line_labels = set(re.findall(r"\[((?:S|C|V|R|L|N)\d+)\]", body_line))
        if line_labels and line_labels.issubset(available):
            # Also verify that if there are multiple sentences in one paragraph, none are dangling uncited claims
            sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9*])", body_line)
            for sent in sentences:
                sent_clean = sent.strip()
                if len(sent_clean) < 12 or any(p in sent_clean.lower() for p in abstention_phrases):
                    continue
                s_labels = set(re.findall(r"\[((?:S|C|V|R|L|N)\d+)\]", sent_clean))
                if not s_labels or not s_labels.issubset(available):
                    return False
        else:
            return False

    return True


_AUTO_CITE_EXTRA_STOPS = {
    "this", "that", "with", "from", "have", "has", "are", "was", "were", "will",
    "would", "should", "could", "their", "there", "which", "what", "when", "where",
    "been", "into", "about", "also", "such", "than", "then", "them", "they",
    "more", "most", "some", "only", "each", "other", "these", "those", "very",
    "much", "many", "your", "you", "its", "it", "and", "the", "for", "not",
    "but", "can", "may", "might", "over", "under", "between", "within", "based",
    "used", "use", "using", "following", "below", "above",
}


def _cite_tokens(text):
    return {
        t for t in re.findall(r"[a-z0-9][a-z0-9._-]{2,}", str(text).lower())
        if len(t) >= 4 and t not in _STOP_WORDS and t not in _AUTO_CITE_EXTRA_STOPS and not t.isdigit()
    }


def _evidence_label_entries(retrieved_docs, case_docs=None, live_docs=None, nvd_docs=None, structured_context=""):
    """Build (label, searchable_text) pairs for every piece of supplied evidence."""
    entries = []
    for i, d in enumerate(live_docs or [], start=1):
        entries.append((f"L{i}", f"{(d.metadata or {}).get('title', '')} {d.page_content or ''}"))
    for i, d in enumerate(nvd_docs or [], start=1):
        entries.append((f"N{i}", f"{(d.metadata or {}).get('title', '')} {d.page_content or ''}"))
    for i, d in enumerate(case_docs or [], start=1):
        entries.append((f"C{i}", f"{(d.metadata or {}).get('title', '')} {d.page_content or ''}"))
    for i, d in enumerate(retrieved_docs or [], start=1):
        entries.append((f"S{i}", f"{(d.metadata or {}).get('title', '')} {d.page_content or ''}"))
    for m in re.finditer(r"\[(V\d+|R\d+)\]\s*\n(.*?)(?=\n\s*\[(?:V|R)\d+\]\s*\n|\Z)", structured_context or "", flags=re.S):
        entries.append((m.group(1), m.group(0)))
    return entries


def _auto_cite_answer(answer, entries):
    """Attach the best-matching evidence label to substantive uncited lines and
    fix dangling sentences inside cited lines. Sentences that share no vocabulary
    with any supplied record are dropped so nothing unsupported is shown."""
    label_re = r"\[(?:S|C|V|R|L|N)\d+\]"
    entry_tokens = [(label, _cite_tokens(text)) for label, text in entries]
    abstention_bits = (
        "not found in the selected datasets", "not reported in the retrieved",
        "no matching", "absent from the supplied", "not explicitly documented",
        "not stated in the retrieved",
    )
    out_lines, dropped = [], 0
    for raw in (answer or "").splitlines():
        line = raw.strip()
        if (not line or line.startswith("#") or line in {"---", "***"}
                or line.lower().startswith("*processing:")
                or re.match(r"^\*\*[^*]+\*\*:?$", line) or line.endswith(":")
                or any(p in line.lower() for p in abstention_bits)):
            out_lines.append(raw)
            continue
        marker_match = re.match(r"^((?:\d+[.)]|[-*•])\s+)(.*)$", line)
        prefix, body = (marker_match.group(1), marker_match.group(2)) if marker_match else ("", line)
        body = body.strip()
        if not body:
            out_lines.append(raw)
            continue

        line_labels = re.findall(label_re, body)
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9*\-])", body) if s.strip()]
        rebuilt, line_dropped = [], 0
        for sent in sentences:
            if len(sent) < 12 or re.search(label_re, sent) or any(p in sent.lower() for p in abstention_bits):
                rebuilt.append(sent)
                continue
            if line_labels:
                # A dangling sentence inside an already cited line inherits the line's first label.
                rebuilt.append(f"{sent} {line_labels[0]}")
                continue
            sent_tokens = _cite_tokens(sent)
            best_label, best_score = None, 0
            for label, tokens in entry_tokens:
                score = len(sent_tokens & tokens)
                if score > best_score:
                    best_label, best_score = label, score
            if best_label and best_score >= 2:
                rebuilt.append(f"{sent} [{best_label}]")
            else:
                line_dropped += 1
        dropped += line_dropped
        if rebuilt:
            out_lines.append(prefix + " ".join(rebuilt))

    # Cosmetic cleanup: remove orphaned headers ("Real Case Study Example:") and
    # truncated enumeration intros ("...all data, including:") left behind when
    # every line of their section was dropped.
    def _is_pure_heading(s):
        s = s.strip()
        if s.startswith("#") or re.match(r"^\*\*[^*]+\*\*:?\s*$", s):
            return True
        return s.endswith(":") and len(s[:-1].split()) <= 8

    final_lines = []
    for idx, raw in enumerate(out_lines):
        s = raw.strip()
        if s:
            nxt = ""
            for j in range(idx + 1, len(out_lines)):
                if out_lines[j].strip():
                    nxt = out_lines[j].strip()
                    break
            if _is_pure_heading(s) and (not nxt or _is_pure_heading(nxt)):
                continue
            if s.endswith(":") and re.search(r"(including|such as|the following|consists? of)\s*$", s[:-1], re.I):
                continue
        final_lines.append(raw)
    return "\n".join(final_lines), dropped


def _synthesise_evidence_answer(query, retrieved_docs, case_docs=None, live_docs=None, nvd_docs=None, structured_context=""):
    """Last-resort grounded answer: quote the strongest retrieved records directly
    instead of presenting an unverifiable model draft."""
    labelled_docs = (
        [(f"C{i}", d) for i, d in enumerate(case_docs or [], start=1)]
        + [(f"L{i}", d) for i, d in enumerate(live_docs or [], start=1)]
        + [(f"N{i}", d) for i, d in enumerate(nvd_docs or [], start=1)]
        + [(f"S{i}", d) for i, d in enumerate(retrieved_docs or [], start=1)]
    )
    query_terms = [t for t in _query_terms(query, max_terms=12) if len(t) > 3]
    bullets = []
    for label, doc in labelled_docs:
        if len(bullets) >= 6:
            break
        text = " ".join(str(doc.page_content or "").split())
        if not text:
            continue
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 40]
        if sentences:
            sentences.sort(key=lambda s: -sum(1 for t in query_terms if t in s.lower()))
            snippet = sentences[0][:420]
        else:
            snippet = text[:420]
        title = (doc.metadata or {}).get("title") or "Retrieved record"
        bullets.append(f"- **{title}:** {snippet} [{label}]")

    if not bullets and structured_context:
        for m in re.finditer(r"\[(V\d+|R\d+)\]\s*\n(.*?)(?=\n\s*\[(?:V|R)\d+\]\s*\n|\Z)", structured_context or "", flags=re.S):
            block = " ".join(m.group(2).split())[:420]
            bullets.append(f"- {block} [{m.group(1)}]")
            if len(bullets) >= 6:
                break

    if not bullets:
        return dataset_only_no_answer(
            "No usable evidence text was retrieved to answer this question.",
            0.0, 0.0
        )
    return (
        "**Evidence-based summary (composed directly from the retrieved records):**\n\n"
        + "\n".join(bullets)
        + "\n\n*Gemma's draft could not be fully citation-verified, so the retrieved "
          "records are quoted directly above. Check the Sources panel for full context.*"
    )


def _ensure_case_study_present(answer, case_docs, live_case_docs, response_style):
    """If the question asked for a documented case but the answer cites no
    case-study evidence, append a compact summary quoted from the top
    retrieved incident record."""
    if response_style not in {"case_studies", "multi_part_case"}:
        return answer
    if re.search(r"\[(?:C|L)\d+\]", answer or ""):
        return answer
    if case_docs:
        doc, label = case_docs[0], "C1"
    elif live_case_docs:
        doc, label = live_case_docs[0], "L1"
    else:
        return answer
    title = (doc.metadata or {}).get("title") or "Documented incident"
    if title.lower() in (answer or "").lower():
        return answer
    text = " ".join(str(doc.page_content or "").split())
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 40]
    snippet = " ".join(sents[:2])[:520] if sents else text[:400]
    if not snippet:
        return answer
    return (answer or "") + f"\n\n**Documented case study — {title}:** {snippet} [{label}]"


# ---------------------------------------------------------------------
# Dataset-level aggregate analytics (deterministic, no Gemma required)
# Answers questions like "in 2026, which sectors were targeted most?" by
# counting values across the loaded structured datasets and returning a
# percentage breakdown plus a chart payload for the website frontend.
# ---------------------------------------------------------------------
from collections import Counter, defaultdict

LAST_CHART_PAYLOAD = None

_AGG_INTENT_RE = re.compile(
    r"\b(most|top|how many|percentage|percent|proportion|distribution|breakdown|"
    r"statistics|stats|count|number of|which (?:sector|industry|country|region|"
    r"group|type|kind|category)|what (?:sectors?|industries|countries|regions?|"
    r"groups?|types?|kinds?|categories|malware)|targeted|utili[sz]ed|popular|"
    r"frequent|common|rank)\b",
    re.I,
)

_AGG_DIMENSION_PATTERNS = [
    ("sector", re.compile(r"\b(sectors?|industr\w+|verticals?)\b", re.I)),
    ("country", re.compile(r"\b(countr\w+|regions?|nations?|geograph\w+)\b", re.I)),
    ("group", re.compile(
        r"\b(ransomware (?:groups?|gangs?|families|actors?)|threat (?:groups?|actors?)|"
        r"groups? (?:claims?|behind|responsible))\b", re.I)),
    ("attack_type", re.compile(
        r"\b(types?|kinds?|families|categories|forms?)\s+of\s+(?:\w+\s+){0,3}"
        r"(malware|attacks?|threats?|cyberattacks?|incidents?)\b|"
        r"\b(malware|attack|threat|cyberattack|incident)\s+(types?|kinds?|families|categories)\b|"
        r"\bmost (?:common|frequent|used|utili[sz]ed|popular|active)\s+(?:types? of )?"
        r"(malware|attacks?|threats?)\b", re.I)),
    ("cve_severity", re.compile(
        r"\b(cvss|severit\w+|kev listed|vulnerabilit\w+|cve records?)\b", re.I)),
]

_ATTACK_TYPE_RULES = [
    ("Ransomware / extortion", r"ransom|extort|encrypt"),
    ("Phishing / social engineering", r"phish|social engineering|credential|bec\b|business email compromise|scam"),
    ("Data breach / leak", r"data breach|data leak|exfiltrat|exposed|leak|breach"),
    ("Supply chain / third-party", r"supply[- ]?chain|third[- ]?party|vendor"),
    ("Vulnerability / zero-day exploitation", r"zero[- ]?day|0[- ]?day|exploit|vulnerab|cve-\d{4}"),
    ("DDoS / disruption", r"\bddos\b|denial[- ]of[- ]service|disrupt"),
    ("Wiper / destructive malware", r"wiper|destruct|destroy|delet"),
    ("Malware / trojan / botnet", r"malware|trojan|backdoor|rat\b|loader|botnet|infosteal|stealer"),
    ("APT / espionage", r"\bapt\b|espionag|state[- ]sponsor|nation[- ]state"),
    ("Web intrusion / compromise", r"deface|webshell|intrusion|unauthori[sz]ed access|compromise"),
]


def detect_aggregate_query(query):
    """Return (dimension, year) when the question asks for a dataset-level
    count/percentage breakdown; otherwise None."""
    q = str(query or "").strip().lower()
    if not q:
        return None
    year_match = re.search(r"\b(20\d{2})\b", q)
    year = year_match.group(1) if year_match else None
    has_intent = bool(_AGG_INTENT_RE.search(q))
    if not year and not has_intent:
        return None
    for dimension, pattern in _AGG_DIMENSION_PATTERNS:
        if pattern.search(q):
            # A bare "what is X / explain X" question with no aggregate intent is
            # a definition question, not a statistics request.
            if not year and re.search(r"^\s*(what is|define|explain)\b", q) and not has_intent:
                return None
            return (dimension, year)
    return None


def _agg_year_filtered_records(df, date_cols, year):
    records = df.to_dict("records") if hasattr(df, "to_dict") else list(df)
    if not year:
        return records
    needle = str(year)
    return [
        row for row in records
        if any(needle in str(row.get(col) or "") for col in date_cols)
    ]


def _agg_display_names(keys):
    """Pick the most common raw spelling for each normalised group key."""
    variants = defaultdict(Counter)
    for raw, norm in keys:
        variants[norm][raw] += 1
    return {norm: variants[norm].most_common(1)[0][0] for norm in variants}


def _norm_agg_label(value):
    text = re.sub(r"\s+", " ", str(value or "")).strip()
    if not text or text.lower() in {"nan", "none", "unknown", "n/a", "-", "null"}:
        return None
    return text.lower()


def _classify_attack_type(title, summary, entities):
    blob = f"{title} {summary} {entities}".lower()
    for label, pattern in _ATTACK_TYPE_RULES:
        if re.search(pattern, blob):
            return label
    return "Other / unclassified"


_AGG_SPECS = {
    "sector": {
        "frame": "ransomware_df", "column": "sector",
        "date_cols": ("discovered", "published"),
        "item_noun": "claims", "item_single": "claim",
        "title": "Ransomware leak-site victim claims by victim sector",
        "dataset": "threatcluster/ransomware-leak-site-victims",
        "caveat": "Leak-site listings are group claims, not independently confirmed breaches.",
    },
    "country": {
        "frame": "ransomware_df", "column": "country",
        "date_cols": ("discovered", "published"),
        "item_noun": "claims", "item_single": "claim",
        "title": "Ransomware leak-site victim claims by country",
        "dataset": "threatcluster/ransomware-leak-site-victims",
        "caveat": "Leak-site listings are group claims, not independently confirmed breaches.",
    },
    "group": {
        "frame": "ransomware_df", "column": "group_name",
        "date_cols": ("discovered", "published"),
        "item_noun": "claims", "item_single": "claim",
        "title": "Ransomware leak-site claims by claiming group",
        "dataset": "threatcluster/ransomware-leak-site-victims",
        "caveat": "Counts are claims posted by each group on its leak site, not confirmed incidents.",
    },
    "attack_type": {
        "frame": "incident_df", "column": None,
        "date_cols": ("first_reported", "last_reported"),
        "item_noun": "incident summaries", "item_single": "incident summary",
        "title": "Incident summaries by attack type (keyword classification)",
        "dataset": "threatcluster/threat-incident-clusters",
        "caveat": "Types are assigned by keyword rules over publisher-generated incident summaries; categories are indicative, not authoritative labels.",
    },
    "cve_severity": {
        "frame": "cve_df", "column": "cvss_v3_severity",
        "date_cols": ("published_date",),
        "item_noun": "CVE records", "item_single": "CVE record",
        "title": "CVE exploitation-signal records by CVSS v3 severity",
        "dataset": "threatcluster/cve-exploitation-signals",
        "caveat": "Severity values are the dataset's recorded CVSS v3 field for this snapshot.",
    },
}


def build_aggregate_answer(query, dimension, year):
    """Compute a count/percentage breakdown directly from a loaded dataset.

    Returns (answer_markdown, stats_context, chart_dict) or None when the
    required dataset or matching rows are unavailable."""
    spec = _AGG_SPECS.get(dimension)
    if not spec:
        return None
    df = globals().get(spec["frame"])
    if df is None or len(df) == 0:
        return None

    filtered = _agg_year_filtered_records(df, spec["date_cols"], year)
    if not filtered:
        return None

    column = spec["column"]
    if dimension != "attack_type" and column not in filtered[0]:
        return None

    pairs = []
    for row in filtered:
        if dimension == "attack_type":
            label = _classify_attack_type(
                as_text(row.get("title"), 300),
                as_text(row.get("summary"), 1500),
                as_text(row.get("entities"), 800),
            )
            pairs.append((label, label))
        else:
            norm = _norm_agg_label(row.get(column))
            if norm:
                pairs.append((str(row.get(column)).strip(), norm))

    if not pairs:
        return None
    names = _agg_display_names(pairs)
    counts = Counter(norm for _, norm in pairs)
    total = sum(counts.values())
    if total == 0:
        return None

    top = counts.most_common(8)
    other_count = total - sum(count for _, count in top)
    items = [
        {"label": names.get(key, key.title()), "count": count, "pct": round(count / total * 100, 1)}
        for key, count in top
    ]
    if other_count > 0:
        items.append({"label": "Other (remaining categories)", "count": other_count, "pct": round(other_count / total * 100, 1)})

    info = _dataset_info(spec["dataset"])
    scope = (
        f"{total:,} {spec['item_noun']} in `{spec['dataset']}`"
        + (f" with a date field containing {year}" if year else " (all dates in the snapshot)")
    )
    bullets = []
    for rank, item in enumerate(items, start=1):
        bar = "█" * max(1, round(item["pct"] / 5))
        noun_txt = spec["item_noun"] if item["count"] != 1 else spec["item_single"]
        bullets.append(
            f"- **{rank}. {item['label']}** — {item['count']:,} {noun_txt} · "
            f"{item['pct']}% {bar}"
        )

    leader = items[0]
    answer = (
        f"**Aggregate analysis (computed directly from the dataset — no model generation):**\n\n"
        f"{scope}.\n\n"
        f"**Top categories:**\n" + "\n".join(bullets) + "\n\n"
        f"**Leading category:** {leader['label']} accounts for {leader['pct']}% "
        f"({leader['count']:,} of {total:,} {spec['item_noun']}).\n\n"
        f"**Method:** counted `{spec['column'] or 'classified attack-type'}` values across the filtered records. "
        f"This is a derived statistic computed at request time, not a single stored record. [AGG1]\n\n"
        f"**Caveat:** {spec['caveat']}"
    )

    stats_context = (
        "[AGG1]\n"
        f"Derived aggregate statistic (not a stored record): counted `{spec['column'] or 'classified attack-type'}` "
        f"across {total:,} records in {spec['dataset']}"
        + (f" filtered to dates containing {year}." if year else " (no date filter).")
        + f" Dataset card: {info.get('url', 'not supplied')} Snapshot generated 2026-09-06."
    )

    chart = {
        "type": "bar",
        "title": spec["title"],
        "dimension": dimension,
        "year": year,
        "total": total,
        "scope": scope,
        "items": items,
        "item_noun": spec["item_noun"],
        "dataset": spec["dataset"],
        "dataset_url": info.get("url", ""),
        "note": spec["caveat"],
    }
    return answer, stats_context, chart


def _build_email_retrieval_query(email_text, findings):
    clean = _refang_text(email_text or "")
    subject_match = re.search(r"(?im)^subject:\s*(.+)$", clean)
    subject = subject_match.group(1).strip() if subject_match else ""
    terms = _query_terms(f"{subject} {clean[:600]}", max_terms=10)
    return (
        f"phishing email social engineering {subject} {' '.join(terms)} {' '.join(findings)}"
    ).strip()[:600]


def process_soc_query(
    user_query,
    analysis_mode,
    use_live_api=False,
    live_search_term=None,
    previous_context=None,
    progress=gr.Progress(track_tqdm=False)
):
    global LAST_CHART_PAYLOAD
    LAST_CHART_PAYLOAD = None
    if not isinstance(user_query, str) or not user_query.strip():
        return "Please enter a CTI question or paste the email text.", "No input was submitted."

    raw_user_query = _clean_injected_frontend_instructions(user_query)
    user_query = (
        resolve_followup_query(raw_user_query, previous_context)
        if analysis_mode != "Email triage"
        else raw_user_query
    )
    total_start = time.perf_counter()
    try:
        progress(0.05, desc="Preparing your request...")
        findings, rule_assessment, case_docs, ioc_lines, ioc_results = [], "", [], [], []
        response_style = query_response_style(user_query) if analysis_mode != "Email triage" else "email_triage"

        if analysis_mode == "Email triage":
            findings = detect_email_indicators(user_query)
            rule_assessment = classify_email_preliminary(user_query, findings)
            retrieval_query = _build_email_retrieval_query(user_query, findings)
        else:
            retrieval_query = user_query[:1500]

        progress(0.18, desc="Searching selected datasets...")
        retrieval_start = time.perf_counter()

        requested_cves, exact_cve_records = (
            retrieve_exact_cve_rows(user_query) if analysis_mode != "Email triage" else ([], [])
        )

        # For CVE IDs missing from the local snapshot, fetch the authoritative
        # record from the NVD live API so any CVE question can still be answered.
        snapshot_cve_ids = {
            str(r.get("_cve_id_norm") or r.get("cve_id") or "").strip().upper()
            for r in exact_cve_records
        }
        nvd_docs, nvd_status = [], ""
        if analysis_mode != "Email triage" and requested_cves:
            missing_cves = [c for c in requested_cves if c not in snapshot_cve_ids]
            if missing_cves:
                nvd_docs, nvd_status = fetch_nvd_cve_records(missing_cves)

        # Only use the pure structured CVE shortcut when the user asked ONLY for basic CVE metadata
        wants_deeper_cve_context = bool(re.search(
            r"\b(miti?gat\w*|mitgat\w*|remediat\w*|how it works|how does|exploit detail|campaign|actor|case|example|workaround|patch steps|defen[cs]\w*|prevent\w*|protect\w*|fix|harden\w*|describ\w*|explain\w*|impact|affects?\b|risks?\b)\b",
            user_query.lower()
        ))
        simple_cve_lookup = (
            bool(requested_cves)
            and (bool(exact_cve_records) or bool(nvd_docs))
            and not use_live_api
            and not wants_deeper_cve_context
        )

        if simple_cve_lookup:
            structured_context = retrieve_structured_context(user_query)
            retrieval_seconds = time.perf_counter() - retrieval_start
            answer_parts = []
            if exact_cve_records:
                snapshot_answer = build_direct_cve_answer(
                    user_query, exact_cve_records,
                    [c for c in requested_cves if c in snapshot_cve_ids],
                    retrieval_seconds, total_start, with_footer=False
                )
                # Keep snapshot text unless it is a pure abstention (then prefer the NVD record).
                if not snapshot_answer.strip().startswith(f"**{_NOT_FOUND}**") or not nvd_docs:
                    answer_parts.append(snapshot_answer)
            if nvd_docs:
                answer_parts.append(build_direct_nvd_answer(user_query, nvd_docs))
            answer = "\n\n".join(answer_parts)
            elapsed = time.perf_counter() - total_start
            answer += (
                f"\n\n*Processing: retrieval {retrieval_seconds:.2f}s · total {elapsed:.2f}s "
                f"(structured/live lookup; Gemma not required).*"
            )
            answer += build_answer_citations(
                [], [], structured_context, live_docs=[], nvd_docs=nvd_docs, answer_text=answer
            )
            if isinstance(previous_context, dict):
                previous_context["last_query"] = raw_user_query
                previous_context["last_answer"] = answer[:800]
            progress(1.0, desc="CVE record lookup complete.")
            return answer, build_sources_panel(
                [], structured_context, live_docs=[], nvd_docs=nvd_docs,
                live_status="Live API skipped: CVE metadata was answered from the selected snapshot and/or the live NVD record.",
                nvd_status=nvd_status or "NVD live fallback not needed: every requested CVE was found in the selected snapshot."
            )

        # Dataset-level aggregate/statistics questions are answered deterministically
        # by counting the loaded structured datasets — no model generation required.
        if analysis_mode != "Email triage":
            aggregate_spec = detect_aggregate_query(user_query)
            if aggregate_spec:
                agg_result = build_aggregate_answer(user_query, *aggregate_spec)
                if agg_result:
                    answer, stats_context, chart_payload = agg_result
                    LAST_CHART_PAYLOAD = chart_payload
                    retrieval_seconds = time.perf_counter() - retrieval_start
                    elapsed = time.perf_counter() - total_start
                    answer += (
                        f"\n\n*Processing: retrieval {retrieval_seconds:.2f}s · total {elapsed:.2f}s "
                        f"(dataset aggregation; Gemma not required).*"
                    )
                    answer += build_answer_citations(
                        [], [], stats_context, live_docs=[], answer_text=answer
                    )
                    if isinstance(previous_context, dict):
                        previous_context["last_query"] = raw_user_query
                        previous_context["last_answer"] = answer[:800]
                    progress(1.0, desc="Dataset aggregate analysis complete.")
                    return answer, build_sources_panel(
                        [], stats_context, live_docs=[],
                        live_status="Live API skipped: the question was answered by aggregating the selected dataset snapshot."
                    )

        if analysis_mode == "Email triage":
            retrieved_docs = hybrid_retrieve(retrieval_query, k=4, candidate_k=12)
            structured_context = "Not applicable to this email-only check."
        else:
            retrieved_docs = hybrid_retrieve(retrieval_query, k=TOP_K, candidate_k=CANDIDATE_K)
            structured_context = retrieve_structured_context(user_query)
            if response_style in {"case_studies", "multi_part_case"}:
                case_docs = retrieve_incident_case_studies(user_query, max_cases=2)

        if analysis_mode == "CTI research":
            live_docs, live_status, live_meta = fetch_threatcluster_live(
                user_query, enabled=bool(use_live_api), privacy_safe_term=live_search_term
            )
        else:
            live_docs, live_status, live_meta = [], "Live API enrichment is disabled until a domain-only lookup is selected below.", {}

        live_case_docs = [d for d in live_docs if (d.metadata or {}).get("bucket") == "clusters"]
        retrieval_seconds = time.perf_counter() - retrieval_start

        context = format_docs(retrieved_docs) if retrieved_docs else "No directly relevant narrative CTI excerpts were retrieved."
        if live_docs:
            context += "\n\nLIVE THREATCLUSTER API RECORDS (distinct live source channel):\n" + format_docs(live_docs, max_chars_per_doc=900, label_prefix="L")
        if nvd_docs:
            context += "\n\nLIVE NVD CVE RECORDS (authoritative lookup for CVEs missing from the selected snapshot):\n" + format_docs(nvd_docs, max_chars_per_doc=900, label_prefix="N")

        case_study_sections = []
        if case_docs:
            case_study_sections.append(format_docs(case_docs, max_chars_per_doc=1400, label_prefix="C"))
        if live_case_docs:
            case_study_sections.append(format_docs(live_case_docs, max_chars_per_doc=1400, label_prefix="L"))
        case_study_context = (
            "\n\n".join(case_study_sections)
            if case_study_sections else "No dedicated incident-cluster records were selected. Use [S#] records if they document a relevant case, or state 'Not found in the selected datasets or live API results.'"
        )

        if (analysis_mode != "Email triage" and response_style == "case_studies"
                and not case_docs and not live_case_docs and not retrieved_docs):
            total_seconds = time.perf_counter() - total_start
            answer = dataset_only_no_answer(
                "No matching named case study or supporting historical report was retrieved.",
                retrieval_seconds, total_seconds
            )
            progress(1.0, desc="No supporting case evidence found.")
            return answer, build_sources_panel(
                retrieved_docs, structured_context, case_docs=case_docs,
                live_docs=live_docs, live_status=live_status,
                nvd_docs=nvd_docs, nvd_status=nvd_status
            )

        if analysis_mode != "Email triage":
            evidence_ok, _ = has_relevant_selected_evidence(
                user_query, retrieved_docs, structured_context, case_docs, live_docs, nvd_docs
            )
            if not evidence_ok:
                total_seconds = time.perf_counter() - total_start
                answer = dataset_only_no_answer(
                    "The retrieved records did not contain enough directly matching evidence.",
                    retrieval_seconds, total_seconds
                )
                progress(1.0, desc="Insufficient matching evidence.")
                return answer, build_sources_panel(
                    retrieved_docs, structured_context, case_docs=case_docs,
                    live_docs=live_docs, live_status=live_status,
                    nvd_docs=nvd_docs, nvd_status=nvd_status
                )

        progress(0.42, desc="Checking indicators and domain reputation...")
        if analysis_mode == "Email triage":
            lookup_start = time.perf_counter()
            ioc_results = check_email_domains(user_query)
            unique_domains = list(dict.fromkeys(r["domain"] for r in ioc_results if r.get("domain")))
            if not use_live_api:
                live_docs, live_status, live_meta = [], "Live API not requested for this email.", {}
            elif unique_domains:
                all_live, statuses = [], []
                for dom in unique_domains[:3]:
                    d_docs, d_status, _ = fetch_threatcluster_live(
                        user_query, enabled=True, privacy_safe_term=dom
                    )
                    all_live.extend(d_docs)
                    statuses.append(d_status)
                live_docs = all_live[:8]
                live_status = " | ".join(statuses)
                if live_docs:
                    context += "\n\nLIVE THREATCLUSTER API RECORDS (domain-only lookup):\n" + format_docs(live_docs, max_chars_per_doc=900, label_prefix="L")
            else:
                live_docs, live_status, live_meta = [], "Live API skipped for email triage: no domain was extracted; email text was not sent.", {}
            retrieval_seconds += time.perf_counter() - lookup_start

            if not ioc_results:
                ioc_lines.append("No URL or sender domain was detected, so no domain-feed lookup was performed. [IOC1]")
            else:
                for result in ioc_results:
                    if result["matches"]:
                        ioc_lines.append(f"MATCH ({result['kind']}): {result['domain']} appears in PhishDestroy blocklist: {', '.join(result['matches'])} [IOC1]")
                    else:
                        ioc_lines.append(f"NO MATCH ({result['kind']}): {result['domain']} was not found in PhishDestroy blocklist; not proof of safety. [IOC1]")

        progress(0.58, desc="Generating a concise, source-bound answer...")
        generation_start = time.perf_counter()
        if analysis_mode == "Email triage":
            answer = email_triage_chain.invoke({
                "indicators": "\n".join(f"- {x}" for x in findings),
                "ioc_results": "\n".join(ioc_lines),
                "feed_timestamp": IOC_FEED_LAST_UPDATED,
                "feed_status": IOC_FEED_STATUS,
                "email_text": user_query[:6000],
                "context": context + "\n\nLive API status: " + live_status,
                "rule_assessment": rule_assessment
            })
        else:
            answer = qa_chain.invoke({
                "context": context,
                "structured_context": structured_context,
                "case_study_context": case_study_context,
                "input": user_query[:1500],
                "response_instruction": response_instruction_for(user_query)
            })

        generation_seconds = time.perf_counter() - generation_start
        if analysis_mode != "Email triage":
            available_labels = (
                {f"S{i}" for i, _ in enumerate(retrieved_docs or [], start=1)}
                | {f"C{i}" for i, _ in enumerate(case_docs or [], start=1)}
                | {f"L{i}" for i, _ in enumerate(live_docs or [], start=1)}
                | {f"N{i}" for i, _ in enumerate(nvd_docs or [], start=1)}
                | set(re.findall(r"\[(V\d+|R\d+)\]", structured_context or ""))
            )
            answer = _normalize_citations(answer, available_labels)

            rescue_entries = _evidence_label_entries(
                retrieved_docs, case_docs, live_docs, nvd_docs, structured_context
            )

            def _rescue_note(dropped_count):
                if dropped_count:
                    return (
                        "\n\n*Note: unsupported statements without matching evidence were removed, "
                        "and citation labels were completed automatically from the retrieved "
                        "evidence; verify each claim in the Sources panel.*"
                    )
                return (
                    "\n\n*Note: citation labels were completed automatically from the retrieved "
                    "evidence; verify each claim in the Sources panel.*"
                )

            citations_ok = _answer_has_complete_valid_citations(
                answer, retrieved_docs, case_docs, structured_context, live_docs, nvd_docs
            )
            if not citations_ok:
                # Fast deterministic rescue first (no extra Gemma call): attach the
                # best-matching evidence label to each uncited claim; drop claims
                # that share no vocabulary with any supplied record.
                rescued, dropped_count = _auto_cite_answer(answer, rescue_entries)
                if _answer_has_complete_valid_citations(
                    rescued, retrieved_docs, case_docs, structured_context, live_docs, nvd_docs
                ):
                    answer = rescued + _rescue_note(dropped_count)
                    citations_ok = True

            if not citations_ok:
                # Only spend a second Gemma generation when the draft could not be rescued.
                repair_instruction = (
                    response_instruction_for(user_query)
                    + "\n\nCITATION REPAIR: Rewrite concisely using ONLY supplied evidence. "
                      "Every factual sentence or bullet MUST end with an exact valid label such as [S1] or [C1]. "
                      "If a requested detail is absent, write 'Not found in the selected datasets or live API results.'"
                )
                repair_start = time.perf_counter()
                repaired = qa_chain.invoke({
                    "context": context,
                    "structured_context": structured_context,
                    "case_study_context": case_study_context,
                    "input": user_query[:1500],
                    "response_instruction": repair_instruction
                        + "\n\nPrevious draft to correct:\n"
                        + str(answer)[:2500]
                })
                generation_seconds += time.perf_counter() - repair_start
                answer = _normalize_citations(repaired, available_labels)

                if not _answer_has_complete_valid_citations(
                    answer, retrieved_docs, case_docs, structured_context, live_docs, nvd_docs
                ):
                    rescued, dropped_count = _auto_cite_answer(answer, rescue_entries)
                    if _answer_has_complete_valid_citations(
                        rescued, retrieved_docs, case_docs, structured_context, live_docs, nvd_docs
                    ):
                        answer = rescued + _rescue_note(dropped_count)
                    else:
                        # Last resort: quote the retrieved records directly instead
                        # of presenting an unverifiable draft.
                        answer = _synthesise_evidence_answer(
                            user_query, retrieved_docs, case_docs, live_docs, nvd_docs, structured_context
                        )

            answer = _ensure_case_study_present(answer, case_docs, live_case_docs, response_style)

        total_seconds = time.perf_counter() - total_start
        answer += build_answer_citations(
            retrieved_docs, case_docs, structured_context,
            include_phish_feed=(analysis_mode == "Email triage"),
            live_docs=live_docs, answer_text=answer, nvd_docs=nvd_docs
        )
        answer += (
            f"\n\n---\n*Processing: retrieval {retrieval_seconds:.2f}s · "
            f"Gemma generation {generation_seconds:.2f}s · total {total_seconds:.2f}s.*"
        )
        if isinstance(previous_context, dict) and analysis_mode != "Email triage":
            previous_context["last_query"] = raw_user_query
            previous_context["last_answer"] = answer[:800]

        print(f"CTI timing | retrieval={retrieval_seconds:.2f}s | generation={generation_seconds:.2f}s | total={total_seconds:.2f}s")
        progress(1.0, desc="Analysis complete.")
        return answer, build_sources_panel(
            retrieved_docs, structured_context, case_docs=case_docs,
            include_phish_feed=(analysis_mode == "Email triage"),
            live_docs=live_docs, live_status=live_status,
            nvd_docs=nvd_docs, nvd_status=nvd_status
        )
    except Exception as exc:
        print(f"CTI application error: {type(exc).__name__}: {exc}")
        return f"**System error:** {type(exc).__name__}: {exc}", "Check the notebook output for the technical traceback."


def _clear_outputs():
    return "", "", "", {}


custom_css = """
:root { color-scheme: dark; }
body, .gradio-container { background: #0b1220 !important; color: #e5e7eb !important; }
.gradio-container { max-width: 1180px !important; }
#hero { background: linear-gradient(120deg,#101b2e,#132b35); border: 1px solid #234454; border-radius: 16px; padding: 22px; }
#hero h1 { color: #65f0c2; margin-bottom: 6px; }
#hero p { color: #cbd5e1; }
#analyze-btn { min-height: 48px; font-weight: 700; }
#result-panel { border: 1px solid #26364a; border-radius: 12px; }
.muted { color: #94a3b8; font-size: 0.9rem; }
"""

app_theme = gr.themes.Soft(primary_hue="emerald", neutral_hue="slate")

with gr.Blocks(title="CTI RAG Assistant") as demo:
    conversation_state = gr.State({})
    gr.Markdown(
        """
        <div id="hero">
        <h1>🛡️ Cyber Threat Intelligence Assistant</h1>
        <p>Evidence-grounded CTI research and preliminary phishing-email triage.</p>
        <p class="muted">Ask natural-language questions about the CTI knowledge base. Retrieval combines semantic vector search with exact-term search; live ThreatCluster enrichment is optional. Every factual answer must cite retrieved evidence. If the selected sources do not support an answer, the assistant abstains rather than using unsupported model memory.</p>
        </div>
        """,
        elem_id="hero"
    )
    with gr.Row():
        gr.Markdown(
            """
            **Knowledge coverage**
            - Historical reports and periodic snapshots
            - Structured CVE and ransomware records
            - Curated CTI Q&A and PhishDestroy domain feed
            - Optional live ThreatCluster incident/entity search
            """
        )
        gr.Markdown(
            """
            **Data freshness**
            - Incident/CVE/ransomware collections: periodic snapshots
            - Historical reports and curated Q&A: archive / curated records
            - PhishDestroy: publisher describes an hourly-updated feed
            - ThreatCluster live API: queried at request time when enabled and keyed
            """
        )
    with gr.Tabs():
        with gr.Tab("🔎 CTI Research"):
            gr.Markdown("Ask about threat actors, malware, vulnerabilities, incidents, targets, impacts and documented mitigations.")
            cti_live_toggle = gr.Checkbox(value=False, label="Include live ThreatCluster API evidence (recent 7-day search; adds latency and may use API credits)")
            cti_input = gr.Textbox(
                label="CTI research question",
                placeholder="Example: Give two documented healthcare ransomware cases and explain their impact and documented response.",
                lines=5
            )
            with gr.Row():
                cti_submit = gr.Button("Analyse CTI question", variant="primary", elem_id="analyze-btn")
                cti_clear = gr.Button("Clear")
            cti_output = gr.Markdown(elem_id="result-panel")
            with gr.Accordion("Sources, dataset citations and retrieved evidence", open=False):
                cti_sources = gr.Markdown()
            gr.Examples(
                examples=[
                    ["What do historical reports state about MiniDuke PDF zero-day attacks and its micro backdoor?"],
                    ["Explain what a wiper attack is in simple terms, provide a real case study example, and give step-by-step defensive mitigations."],
                    ["Give two documented healthcare ransomware cases and explain their impact and documented response."]
                ],
                inputs=cti_input
            )
        with gr.Tab("✉️ Email Triage"):
            gr.Markdown("Paste the email subject, sender, body and visible URLs. Remove confidential or personal information before submitting.")
            email_live_toggle = gr.Checkbox(value=False, label="Optional: send extracted domain only to ThreatCluster live API (email text is never sent)")
            email_input = gr.Textbox(
                label="Email content",
                placeholder="Subject: ...\nFrom: ...\n\nPaste the email body and URLs here...",
                lines=9
            )
            with gr.Row():
                email_submit = gr.Button("Analyse email", variant="primary", elem_id="analyze-btn")
                email_clear = gr.Button("Clear")
            email_output = gr.Markdown(elem_id="result-panel")
            with gr.Accordion("Email indicators, feed status and supporting CTI evidence", open=False):
                email_sources = gr.Markdown()
            gr.Examples(
                examples=[
                    ["Subject: Urgent account suspension\nFrom: security@example-mail.com\n\nYour account will be closed today. Verify your password immediately using https://example.invalid/verify."]
                ],
                inputs=email_input
            )

    def run_cti_query(q, live_toggle, state, progress=gr.Progress(track_tqdm=False)):
        state = dict(state or {})
        ans, src = process_soc_query(q, "CTI research", use_live_api=live_toggle, previous_context=state, progress=progress)
        return ans, src, state

    def run_email_triage(q, live_toggle, progress=gr.Progress(track_tqdm=False)):
        return process_soc_query(q, "Email triage", use_live_api=live_toggle, progress=progress)

    cti_submit.click(
        fn=run_cti_query,
        inputs=[cti_input, cti_live_toggle, conversation_state],
        outputs=[cti_output, cti_sources, conversation_state],
        show_progress="full", concurrency_limit=1, concurrency_id="gemma-gpu"
    )
    email_submit.click(
        fn=run_email_triage,
        inputs=[email_input, email_live_toggle],
        outputs=[email_output, email_sources],
        show_progress="full", concurrency_limit=1, concurrency_id="gemma-gpu"
    )
    cti_clear.click(_clear_outputs, outputs=[cti_input, cti_output, cti_sources, conversation_state])
    email_clear.click(lambda: ("", "", ""), outputs=[email_input, email_output, email_sources])

demo.queue(default_concurrency_limit=1, max_size=16)
if os.getenv("SPACE_ID"):
    demo.launch(theme=app_theme, css=custom_css, server_name="0.0.0.0", server_port=7860, share=False, debug=False)
else:
    # share=True prints a temporary public https://*.gradio.live link so anyone
    # re-running this notebook (e.g. a marker) gets a real web interface without
    # needing the Vercel deployment or any access keys. The link lives ~72 hours.
    demo.launch(theme=app_theme, css=custom_css, share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bcca8df3ca8305f407.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Retrieval reliability update (2026-11-23)

**Important CVE fallback correction:** an exact CVE identifier missing from the structured CVE snapshot no longer causes an immediate "not found" response. The pipeline now falls back to the **NIST National Vulnerability Database (NVD) live API** — a free, authoritative, keyless source — and injects the record as `[N#]` evidence. This means *any* real CVE can be answered, not only CVEs present in the 2026-09-06 snapshot.

**Answer reliability update:** Gemma-2-2b frequently fails the strict "every sentence must carry a valid citation label" check even when its draft is mostly grounded. Previously the answer was withheld after a single repair attempt. The pipeline now performs a deterministic grounding rescue:

1. One constrained citation-repair generation (unchanged).
2. If still invalid, uncited sentences are auto-labelled with the best-matching retrieved record, and sentences sharing no vocabulary with any supplied record are removed so nothing unsupported is shown.
3. If no verifiable draft remains, a grounded summary is composed by quoting the retrieved records directly with their labels.

Answers are therefore produced whenever evidence exists; the assistant still abstains only when no dataset, NVD, or live record matches the question.

**Typo tolerance:** follow-up phrasings such as "mitgation for that?" are now recognised as mitigation intent.

**Dataset aggregate analytics:** statistics-style questions (for example *"in 2026, which sectors were targeted most?"* or *"what types of malware were utilised most in 2026?"*) are now answered by directly counting the loaded structured datasets — victim `sector`/`country`/`group_name` fields from the ransomware leak-site snapshot, CVSS severity from the CVE snapshot, and keyword-classified attack types from incident summaries. The result is a deterministic percentage breakdown with an `[AGG#]` derived-statistic citation, plus a `chart` payload the website renders as a bar chart. No model generation is involved, so these answers are fast and fully reproducible.

The existing Email Triage flow, PhishDestroy domain checks, and API endpoints are retained.


# Extended Evaluation: RAG vs. No-Retrieval Baseline (Ablation)

The same Gemma 2 2B model answers each question twice: once through the full RAG pipeline (`process_soc_query`) and once with no retrieved evidence. Comparing latency, citation use and abstention isolates what retrieval contributes. The question set covers historical, vulnerability, ransomware, definition/mitigation, case-study and out-of-scope questions; the out-of-scope question tests whether the system abstains instead of hallucinating.

Automatic columns (latency, citation count, abstention) are measured. Correctness, grounding and relevance must be scored manually (0–2) after checking each claim against the cited records. Do not invent scores.


In [10]:
# ---------------------------------------------------------------------
# Extended evaluation: full RAG pipeline vs. no-retrieval Gemma baseline
# ---------------------------------------------------------------------
# Run after the Gradio cell (it reuses process_soc_query). The baseline sends the
# same question to the same Gemma 2 2B model WITHOUT retrieved evidence, so the
# difference isolates the contribution of retrieval (an ablation).
import re
import time
import pandas as pd

EVAL_QUESTIONS = [
    ("historical", "What do historical reports state about MiniDuke PDF zero-day attacks and its micro backdoor?"),
    ("vulnerability", latest_cve_question if "latest_cve_question" in globals() else "What vulnerability fields are available in the loaded CVE snapshot?"),
    ("ransomware", "What ransomware-group victim claims are recorded for healthcare organisations, and what does the dataset actually establish?"),
    ("definition_mitigation", "What is a wiper attack and what mitigations are documented in the selected datasets?"),
    ("case_studies", "Give two documented healthcare ransomware incident cases. Include actual losses only when reported."),
    ("out_of_scope", "What was the exact financial loss of the 2031 Mars colony ransomware attack?"),
]

BASELINE_TEMPLATE = (
    "<start_of_turn>user\nYou are a Cyber Threat Intelligence analyst. Answer concisely.\n\n"
    "{question}<end_of_turn>\n<start_of_turn>model\n"
)
CITATION_PATTERN = re.compile(r"\[(?:S|C|V|R|L|N)\d+\]")
NOT_FOUND = "not found in the selected datasets or live api results"
RETRIEVAL_PATTERN = re.compile(r"retrieval\s+([\d.]+)s")
GENERATION_PATTERN = re.compile(r"Gemma generation\s+([\d.]+)s")


def _no_progress(*args, **kwargs):
    return None


def _answer_body(answer):
    # Strip the appended citation list and timing footer so only Gemma's text is scored.
    return re.split(r"\n\n### Dataset citations|\n\n---\n\*Processing", answer or "", maxsplit=1)[0]


ablation_rows = []
for category, question in EVAL_QUESTIONS:
    rag_start = time.perf_counter()
    rag_answer, _sources = process_soc_query(question, "CTI research", use_live_api=False, progress=_no_progress)
    rag_seconds = time.perf_counter() - rag_start
    retrieval_match = RETRIEVAL_PATTERN.search(rag_answer or "")
    generation_match = GENERATION_PATTERN.search(rag_answer or "")

    base_start = time.perf_counter()
    baseline_answer = llm.invoke(BASELINE_TEMPLATE.format(question=question))
    baseline_seconds = time.perf_counter() - base_start

    rag_body = _answer_body(rag_answer)
    ablation_rows.append({
        "category": category,
        "question": question,
        "rag_retrieval_seconds": float(retrieval_match.group(1)) if retrieval_match else None,
        "rag_generation_seconds": float(generation_match.group(1)) if generation_match else None,
        "rag_total_seconds": round(rag_seconds, 3),
        "baseline_seconds": round(baseline_seconds, 3),
        "rag_citations": len(CITATION_PATTERN.findall(rag_body)),
        "baseline_citations": len(CITATION_PATTERN.findall(baseline_answer)),
        "rag_abstained": NOT_FOUND in rag_body.lower(),
        "baseline_abstained": NOT_FOUND in baseline_answer.lower(),
        "rag_answer": rag_answer,
        "baseline_answer": baseline_answer,
        # Manual scores (0-2). Fill these in after checking each claim against the cited evidence.
        "rag_correctness_0_to_2": None,
        "rag_grounding_0_to_2": None,
        "rag_relevance_0_to_2": None,
        "baseline_correctness_0_to_2": None,
        "baseline_grounding_0_to_2": None,
        "baseline_relevance_0_to_2": None,
        "review_notes": "",
    })
    print(f"[{category}] RAG {rag_seconds:.2f}s · baseline {baseline_seconds:.2f}s")

ablation_results = pd.DataFrame(ablation_rows)
summary_columns = [
    "category", "rag_retrieval_seconds", "rag_generation_seconds", "rag_total_seconds",
    "baseline_seconds", "rag_citations", "baseline_citations", "rag_abstained", "baseline_abstained",
]
display(ablation_results[summary_columns])
print("\nMean latency (s):", ablation_results[["rag_total_seconds", "baseline_seconds"]].mean().round(2).to_dict())
ablation_results.to_csv("cti_rag_vs_baseline_evaluation.csv", index=False)
print("Saved cti_rag_vs_baseline_evaluation.csv. Fill in the manual 0-2 scores before reporting results.")


CTI timing | retrieval=1.97s | generation=52.59s | total=54.60s
[historical] RAG 54.60s · baseline 16.64s
[vulnerability] RAG 0.01s · baseline 15.57s
CTI timing | retrieval=1.62s | generation=12.28s | total=13.91s
[ransomware] RAG 13.91s · baseline 43.13s


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


CTI timing | retrieval=2.66s | generation=113.37s | total=116.04s
CTI timing | retrieval=0.51s | generation=92.09s | total=92.63s
[definition_mitigation] RAG 92.63s · baseline 18.16s
CTI timing | retrieval=1.62s | generation=9.85s | total=11.48s
[case_studies] RAG 11.48s · baseline 17.19s
CTI timing | retrieval=2.11s | generation=8.57s | total=10.69s
[out_of_scope] RAG 10.69s · baseline 9.79s


,category,rag_retrieval_seconds,rag_generation_seconds,rag_total_seconds,baseline_seconds,rag_citations,baseline_citations,rag_abstained,baseline_abstained
0,historical,1.97,52.59,54.599,16.638,14,0,False,False
1,vulnerability,0.00,NaN,0.014,15.569,4,0,False,False
2,ransomware,1.62,12.28,13.911,43.130,8,0,False,False
3,definition_mitigation,0.51,92.09,92.631,18.163,22,0,False,False
4,case_studies,1.62,9.85,11.477,17.189,5,0,True,False
5,out_of_scope,2.11,8.57,10.693,9.794,12,0,False,False



Mean latency (s): {'rag_total_seconds': 30.55, 'baseline_seconds': 20.08}
Saved cti_rag_vs_baseline_evaluation.csv. Fill in the manual 0-2 scores before reporting results.


# Serve the RAG Pipeline for the Vercel Website — Stable ngrok Endpoint

Run this cell after the RAG pipeline and Gradio setup have completed. It starts the existing FastAPI API on port 8000 and exposes it through the fixed ngrok developer domain assigned to this account: `https://gathering-unusual-shininess.ngrok-free.dev`.

The cell reads `CTI_ACCESS_KEY` and `NGROK_AUTHTOKEN` from Google Colab Secrets. It tests the local API and then tests the public HTTPS API. Both tests must pass with HTTP 200 before updating Cloud Run's `COLAB_BACKEND_URL`.

The ngrok hostname remains the same when the ngrok agent is restarted, so Cloud Run and Vercel do not need a URL update each time. However, the Colab runtime and notebook server must remain active; the stable hostname does not keep Colab running after a runtime disconnect. Do not paste either secret into notebook code or GitHub.

In [11]:
# ---------------------------------------------------------------------
# Serve the RAG pipeline as a JSON API for the Vercel website
# Stable ngrok developer-domain tunnel (replaces Cloudflare Quick Tunnel)
# Run after the RAG pipeline and Gradio setup cells have completed.
# ---------------------------------------------------------------------

import inspect
import os
import re
import secrets
import shutil
import socket
import subprocess
import threading
import time
from datetime import datetime, timezone

import requests
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

API_PORT = int(os.getenv("CTI_API_PORT", "8000"))
NGROK_DOMAIN = "gathering-unusual-shininess.ngrok-free.dev"
NGROK_URL = f"https://{NGROK_DOMAIN}"


def _read_secret(name):
    value = os.getenv(name)
    if value:
        return value.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        return value.strip() if value else None
    except Exception:
        return None


# Read both keys from Colab Secrets. Never print either secret.
CTI_ACCESS_KEY = _read_secret("CTI_ACCESS_KEY")
NGROK_AUTHTOKEN = _read_secret("NGROK_AUTHTOKEN")

_MISSING_SECRETS = [
    name for name, value in
    (("CTI_ACCESS_KEY", CTI_ACCESS_KEY), ("NGROK_AUTHTOKEN", NGROK_AUTHTOKEN))
    if not value
]

if _MISSING_SECRETS:
    print("Skipping public API serving: missing secrets (Colab Secrets / Kaggle Secrets) ->", ", ".join(_MISSING_SECRETS))
    print("The Gradio interface above still works for local evaluation (share=True prints a public link).")
    print("To serve the public API for the deployed website, add these secrets and re-run this cell.")
else:

    _gpu_lock = threading.Lock()
    _SESSION_CONTEXT: dict[str, dict[str, str]] = {}

    _TIMING_PATTERNS = {
        "retrieval_seconds": re.compile(r"retrieval\s+([\d.]+)s"),
        "generation_seconds": re.compile(r"Gemma generation\s+([\d.]+)s"),
        "total_seconds": re.compile(r"total\s+([\d.]+)s"),
    }


    def _parse_timings(answer):
        found = {}
        for name, pattern in _TIMING_PATTERNS.items():
            match = pattern.search(answer or "")
            found[name] = float(match.group(1)) if match else None
        return found


    class QueryRequest(BaseModel):
        question: str = Field(..., min_length=1, max_length=8000)
        mode: str = Field("cti", pattern="^(cti|email)$")
        use_live_api: bool = False
        session_id: str | None = Field(default="default", max_length=128)
        previous_question: str | None = Field(default=None, max_length=2000)
        previous_answer: str | None = Field(default=None, max_length=4000)


    api = FastAPI(title="CTI RAG Assistant API", version="3.1")
    api.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],
        allow_methods=["GET", "POST", "OPTIONS"],
        allow_headers=["Content-Type", "X-Access-Key", "X-Gateway-Key"],
    )


    def _check_key(provided_key):
        if not secrets.compare_digest(str(provided_key or ""), CTI_ACCESS_KEY):
            raise HTTPException(status_code=401, detail="Invalid or missing access key.")


    @api.get("/api/health")
    def health(
        x_access_key: str | None = Header(default=None),
        x_gateway_key: str | None = Header(default=None),
    ):
        _check_key(x_access_key or x_gateway_key)
        return {
            "status": "ok",
            "model": globals().get("model_id", "google/gemma-2-2b-it"),
            "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
            "top_k": globals().get("TOP_K"),
            "candidate_k": globals().get("CANDIDATE_K"),
            "max_new_tokens": globals().get("MAX_NEW_TOKENS"),
            "vector_chunks": len(globals().get("docs", []) or []),
            "cve_records": int(len(cve_df)) if "cve_df" in globals() else None,
            "ransomware_claims": int(len(ransomware_df)) if "ransomware_df" in globals() else None,
            "phish_domains": len(globals().get("IOC_DOMAINS", set()) or set()),
            "phish_feed_status": globals().get("IOC_FEED_STATUS"),
            "phish_feed_updated": globals().get("IOC_FEED_LAST_UPDATED"),
            "live_api_configured": bool(get_threatcluster_api_key()) if "get_threatcluster_api_key" in globals() else False,
            "server_time_utc": datetime.now(timezone.utc).isoformat(),
        }


    @api.post("/api/query")
    def query(
        body: QueryRequest,
        x_access_key: str | None = Header(default=None),
        x_gateway_key: str | None = Header(default=None),
    ):
        _check_key(x_access_key or x_gateway_key)
        analysis_mode = "Email triage" if body.mode == "email" else "CTI research"
        sid = (body.session_id or "default").strip() or "default"

        if body.previous_question:
            prev_ctx = {
                "last_query": body.previous_question,
                "last_answer": body.previous_answer or "",
            }
        else:
            prev_ctx = dict(_SESSION_CONTEXT.get(sid, {}))

        supports_context = "previous_context" in inspect.signature(process_soc_query).parameters
        started = time.perf_counter()

        with _gpu_lock:
            if supports_context:
                answer, sources = process_soc_query(
                    body.question,
                    analysis_mode,
                    use_live_api=body.use_live_api,
                    previous_context=prev_ctx,
                    progress=lambda *a, **k: None,
                )
            else:
                answer, sources = process_soc_query(
                    body.question,
                    analysis_mode,
                    use_live_api=body.use_live_api,
                    progress=lambda *a, **k: None,
                )

        if analysis_mode == "CTI research" and prev_ctx.get("last_query"):
            _SESSION_CONTEXT[sid] = prev_ctx

        # Optional chart payload produced by the dataset aggregate-analysis path.
        chart = globals().get("LAST_CHART_PAYLOAD")

        return {
            "mode": body.mode,
            "answer_markdown": answer,
            "sources_markdown": sources,
            "chart": chart,
            "timings": {**_parse_timings(answer), "server_seconds": round(time.perf_counter() - started, 3)},
        }


    def _port_is_open(port):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(0.5)
            return sock.connect_ex(("127.0.0.1", port)) == 0


    # Stop a previous API server created by this cell before rebinding port 8000.
    _previous_server = globals().get("api_server")
    if _previous_server is not None and getattr(_previous_server, "started", False):
        print("Stopping the previous API server...")
        _previous_server.should_exit = True
        for _ in range(30):
            if not _port_is_open(API_PORT):
                break
            time.sleep(0.5)

    if _port_is_open(API_PORT):
        raise RuntimeError(
            f"Port {API_PORT} is already occupied. The previous API did not stop cleanly. "
            "Use Runtime > Restart session, then run the notebook setup cells again."
        )


    def _start_api_server():
        config = uvicorn.Config(api, host="0.0.0.0", port=API_PORT, log_level="info", access_log=True)
        server = uvicorn.Server(config)
        thread = threading.Thread(target=server.run, name="cti-fastapi-server", daemon=True)
        thread.start()
        for _ in range(60):
            if server.started:
                return server, thread
            if not thread.is_alive():
                raise RuntimeError("Uvicorn stopped during startup. Check the preceding Colab error.")
            time.sleep(0.25)
        server.should_exit = True
        raise RuntimeError("FastAPI did not start listening on port 8000 within 15 seconds.")


    api_server, api_server_thread = _start_api_server()
    LOCAL_BACKEND_URL = f"http://127.0.0.1:{API_PORT}"
    _health_headers = {"X-Access-Key": CTI_ACCESS_KEY}

    # Test FastAPI locally before exposing it through ngrok.
    try:
        local_test = requests.get(
            f"{LOCAL_BACKEND_URL}/api/health", headers=_health_headers, timeout=10
        )
        local_test.raise_for_status()
        print(f"LOCAL API TEST: PASS (HTTP {local_test.status_code})")
        print("Local RAG health check passed.")
    except Exception as exc:
        raise RuntimeError(
            "LOCAL API TEST FAILED. The issue is inside the Colab FastAPI server, "
            f"before ngrok is involved. Details: {type(exc).__name__}: {exc}"
        ) from exc


    # Install the official ngrok Linux binary if it is not already available.
    def _ensure_ngrok_binary():
        binary = shutil.which("ngrok")
        if binary:
            return binary

        print("Installing the ngrok agent in this Colab runtime...")
        install_command = (
            "set -o pipefail; "
            "curl -fsSL https://bin.ngrok.com/c/bNyj1mQVY4c/"
            "ngrok-v3-stable-linux-amd64.tgz | tar -xz -C /usr/local/bin"
        )
        subprocess.run(["bash", "-lc", install_command], check=True)
        binary = shutil.which("ngrok")
        if not binary:
            raise RuntimeError("ngrok installation completed, but the ngrok executable was not found.")
        return binary


    ngrok_binary = _ensure_ngrok_binary()

    # Store the token in ngrok's runtime configuration without displaying it.
    _token_result = subprocess.run(
        [ngrok_binary, "config", "add-authtoken", NGROK_AUTHTOKEN],
        capture_output=True,
        text=True,
    )
    if _token_result.returncode != 0:
        raise RuntimeError(
            "ngrok could not save the authentication token. Check that NGROK_AUTHTOKEN "
            "in Colab/Kaggle Secrets contains your ngrok authtoken. The token was not printed."
        )

    # Stop a previous ngrok process created by this cell, if one exists.
    _previous_ngrok = globals().get("ngrok_process")
    if _previous_ngrok is not None and _previous_ngrok.poll() is None:
        print("Stopping the previous ngrok agent...")
        _previous_ngrok.terminate()
        try:
            _previous_ngrok.wait(timeout=8)
        except subprocess.TimeoutExpired:
            _previous_ngrok.kill()
            _previous_ngrok.wait(timeout=5)

    # Stop the old Cloudflare Quick Tunnel if this cell was run previously.
    _previous_cloudflare = globals().get("tunnel_process")
    if _previous_cloudflare is not None and _previous_cloudflare.poll() is None:
        _previous_cloudflare.terminate()
        try:
            _previous_cloudflare.wait(timeout=5)
        except subprocess.TimeoutExpired:
            _previous_cloudflare.kill()

    # Start ngrok using the static developer domain assigned to this account.
    ngrok_log_path = "/tmp/cti-ngrok-agent.log"
    ngrok_log_file = open(ngrok_log_path, "w")
    ngrok_process = subprocess.Popen(
        [ngrok_binary, "http", str(API_PORT), "--url", NGROK_URL],
        stdout=ngrok_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    # Wait for the local ngrok agent API to confirm that the assigned HTTPS domain is online.
    NGROK_AGENT_API = "http://127.0.0.1:4040/api/tunnels"
    endpoint_online = False
    last_agent_error = None
    for attempt in range(60):
        if ngrok_process.poll() is not None:
            break
        try:
            agent_response = requests.get(NGROK_AGENT_API, timeout=2)
            agent_response.raise_for_status()
            tunnels = agent_response.json().get("tunnels", [])
            if any(tunnel.get("public_url", "").rstrip("/") == NGROK_URL for tunnel in tunnels):
                endpoint_online = True
                break
        except Exception as exc:
            last_agent_error = f"{type(exc).__name__}: {exc}"
        time.sleep(1)

    if not endpoint_online:
        try:
            ngrok_log_file.flush()
            with open(ngrok_log_path, "r") as log_file:
                ngrok_log = log_file.read()[-5000:]
        except Exception:
            ngrok_log = "Could not read the ngrok agent log."
        raise RuntimeError(
            "ngrok did not bring the assigned endpoint online. "
            "Check the domain spelling, ngrok account status, and NGROK_AUTHTOKEN.\n"
            f"Agent API detail: {last_agent_error or 'ngrok process exited'}\n"
            f"Recent ngrok log:\n{ngrok_log}"
        )

    print("\nNGROK ENDPOINT ONLINE:", NGROK_URL)

    # Test the real public HTTPS route using the same backend access-key header.
    public_test = None
    public_error = None
    for attempt in range(1, 11):
        try:
            public_test = requests.get(
                f"{NGROK_URL}/api/health", headers=_health_headers, timeout=20
            )
            if public_test.status_code == 200:
                break
            public_error = f"HTTP {public_test.status_code}: {public_test.text[:500]}"
        except Exception as exc:
            public_error = f"{type(exc).__name__}: {exc}"
        time.sleep(3)

    if public_test is not None and public_test.status_code == 200:
        print("NGROK PUBLIC API TEST: PASS (HTTP 200)")
        print("Public RAG health check passed.")
        print("\nSTABLE COLAB BACKEND URL:", NGROK_URL)
        print("Use this base URL in Cloud Run COLAB_BACKEND_URL (no trailing slash).")
        print("The hostname stays the same after restarting ngrok; Colab must still be running.")
    else:
        print("NGROK PUBLIC API TEST: FAILED")
        print("Last error:", public_error or "No response received")
        print("Do not update Cloud Run until this public test returns HTTP 200.")

    # Flush the log handle after the agent process has been started.
    ngrok_log_file.flush()


INFO:     Started server process [418]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:60582 - "GET /api/health HTTP/1.1" 200 OK
LOCAL API TEST: PASS (HTTP 200)
Local RAG health check passed.
Installing the ngrok agent in this Colab runtime...

NGROK ENDPOINT ONLINE: https://gathering-unusual-shininess.ngrok-free.dev
INFO:     34.50.185.37:0 - "GET /api/health HTTP/1.1" 200 OK
NGROK PUBLIC API TEST: PASS (HTTP 200)
Public RAG health check passed.

STABLE COLAB BACKEND URL: https://gathering-unusual-shininess.ngrok-free.dev
Use this base URL in Cloud Run COLAB_BACKEND_URL (no trailing slash).
The hostname stays the same after restarting ngrok; Colab must still be running.


##Baseline Testing: Gemma 2 2B Without Retrieval

In [14]:

# ============================================================
# BASELINE TEST: GEMMA 2 2B WITHOUT RAG RETRIEVAL
# No ChromaDB, no BM25, no retrieved CTI documents.
# Uses the same loaded Gemma model as the RAG system.
# ============================================================

import time
import pandas as pd
from IPython.display import display, Markdown

# ------------------------------------------------------------
# 1. Enter your test question
# ------------------------------------------------------------

question = (
    "What is CVE-2021-26855, and which Microsoft Exchange vulnerability does it describe? "
)

# You can change the question above whenever you want.

# ------------------------------------------------------------
# 2. Create a baseline prompt
# ------------------------------------------------------------

BASELINE_PROMPT = (
    "<start_of_turn>user\n"
    "You are a Cyber Threat Intelligence assistant.\n"
    "Answer the user's question clearly and in simple language.\n"
    "Use your existing learned knowledge only.\n"
    "You have NOT searched any external documents or CTI datasets.\n"
    "Do not invent citations, sources, statistics or incident details.\n"
    "If you are uncertain about a specific fact, say so.\n\n"
    "Question: {question}\n"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)

prompt = BASELINE_PROMPT.format(question=question)

# ------------------------------------------------------------
# 3. Generate answer WITHOUT retrieval
# ------------------------------------------------------------

print("=" * 70)
print("GEMMA 2 2B — BASE MODEL (NO RETRIEVAL)")
print("=" * 70)

print("\nQuestion:")
print(question)

print("\nGenerating answer...")
start_time = time.perf_counter()

baseline_answer = llm.invoke(prompt)

baseline_seconds = time.perf_counter() - start_time

# ------------------------------------------------------------
# 4. Display the generated answer
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASE MODEL ANSWER")
print("=" * 70)

display(Markdown(str(baseline_answer)))

# ------------------------------------------------------------
# 5. Display baseline performance
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASELINE PERFORMANCE")
print("=" * 70)

baseline_results = pd.DataFrame([{
    "Model": "Gemma 2 2B",
    "Retrieval": "Disabled",
    "Embedding Search": "No",
    "ChromaDB": "No",
    "BM25": "No",
    "Response Time (seconds)": round(baseline_seconds, 3),
    "Generated Answer Length": len(str(baseline_answer))
}])

display(baseline_results)

print("\nBaseline test completed.")
print("Important: This answer was generated without retrieved CTI evidence.")


GEMMA 2 2B — BASE MODEL (NO RETRIEVAL)

Question:
What is CVE-2021-26855, and which Microsoft Exchange vulnerability does it describe? 

Generating answer...

BASE MODEL ANSWER


CVE-2021-26855 is a vulnerability in Microsoft Exchange Server that allows attackers to take control of the server. 

This vulnerability affects versions of Microsoft Exchange Server from 2013 to 2019. 

It's important to note that this vulnerability is related to how the server handles email messages. 



BASELINE PERFORMANCE


,Model,Retrieval,Embedding Search,ChromaDB,BM25,Response Time (seconds),Generated Answer Length
0,Gemma 2 2B,Disabled,No,No,No,10.739,305



Baseline test completed.
Important: This answer was generated without retrieved CTI evidence.
